<a href="https://colab.research.google.com/github/Abmanna/Abmanna-Omniscient-Multi-Agent-Drug-Discovery-OS-Godly-Generative-Scaffold-Approach/blob/main/Pathogen_multitarget_screening_gnina_forms_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-organism, multi-target virtual screening with GNINA - one form per step

**Workflow (each step has its own form, opened under its cell):**
1. **Targets** - pick an organism (bacteria, virus, protozoa, *Plasmodium*, amoeba, fungi, helminths, human, or any taxon); retrieve its proteins from the RCSB PDB; keep only well-resolved, ligand-bound, density-supported structures.
2. **Validate** - prepare each structure and re-dock its own crystal ligand (RMSD + PoseBusters). Only targets that reproduce the crystal pose are used.
3. **Library** - upload SMILES / CSV / SDF / MOL2 / XLSX, paste SMILES, or pull from ChEMBL / PubChem / a URL.
4. **Dock** - GNINA on every target; scores are relative to each target's crystal ligand.
5. **Shortlist** - rank with drug-likeness and PoseBusters checks.
6. **Poses** - cut-offs, every molecule x protein pair in a table, residue hot-spots, 2D and 3D view of any pair.
7. **SAR** - R-group table, Free-Wilson model, matched pairs, activity cliffs, and enumeration of predicted-more-potent analogs.
8. **ADME** - rule-based drug-likeness, solubility, absorption/CNS map, structural alerts (optional ML panel).
9. **Export** - one Excel workbook.

**How to run:** run the cells **one after the other** (not *Run all*) and use the form that appears under each step. A T4 GPU runtime makes docking ~5x faster. To skip the forms and run everything with the settings of cell 0, set `MODE` to *run everything*.

> Docking/CNN scores prioritise hypotheses; they do not predict IC50 or MIC. SAR and ADME here are screening filters and simple models - use them to choose compounds to test.

In [ ]:
import subprocess, sys
def sh(cmd, check=True):
    return subprocess.run(cmd, shell=True, check=check, capture_output=True, text=True)
sh(f"{sys.executable} -m pip -q install rdkit biopython py3Dmol seaborn tqdm openpyxl scikit-learn requests posebusters gradio")
sh("apt-get -qq update && apt-get -qq install -y openbabel", check=False)
print('ready')

In [ ]:
#@title 0. SETTINGS - organism, ligand library, cut-offs { display-mode: "form" }
#@markdown ### How to run
MODE = "forms (one per step)" #@param ["forms (one per step)", "run everything with the settings below"]
UI_PASSWORD = "" #@param {type:"string"}
#@markdown *(Colab publishes forms through a temporary public gradio.live link. Set a password if you share the session; the login name is `screening`.)*
#@markdown ### Target organism
ORGANISM = "Protozoa | Leishmania (leishmaniasis)" #@param ["Protozoa | Leishmania (leishmaniasis)", "Protozoa | Trypanosoma cruzi (Chagas disease)", "Protozoa | Trypanosoma brucei (sleeping sickness)", "Protozoa | Plasmodium falciparum (malaria)", "Protozoa | Plasmodium vivax (malaria)", "Protozoa | Plasmodium - any species", "Protozoa | Toxoplasma gondii", "Protozoa | Cryptosporidium", "Protozoa | Giardia", "Protozoa | Trichomonas vaginalis", "Amoeba | Entamoeba histolytica (amoebiasis)", "Amoeba | Naegleria fowleri (brain-eating amoeba)", "Amoeba | Acanthamoeba", "Bacteria | Mycobacterium tuberculosis", "Bacteria | Mycobacterium - any species", "Bacteria | Staphylococcus aureus (incl. MRSA)", "Bacteria | Escherichia coli", "Bacteria | Pseudomonas aeruginosa", "Bacteria | Klebsiella pneumoniae", "Bacteria | Acinetobacter baumannii", "Bacteria | Helicobacter pylori", "Bacteria | Streptococcus pneumoniae", "Bacteria | Enterococcus faecalis", "Bacteria | Salmonella", "Bacteria | Vibrio cholerae", "Bacteria | Clostridioides difficile", "Bacteria | Neisseria gonorrhoeae", "Bacteria | Bacillus anthracis", "Bacteria | Burkholderia pseudomallei", "Bacteria | ALL bacteria (very large - raise MAX_TARGETS / use keywords)", "Virus | SARS-CoV-2", "Virus | HIV-1", "Virus | Hepatitis C virus", "Virus | Hepatitis B virus", "Virus | Influenza A virus", "Virus | Dengue virus", "Virus | Zika virus", "Virus | Ebola virus (Zaire)", "Virus | Chikungunya virus", "Virus | Herpes simplex virus 1", "Virus | ALL viruses (very large - use keywords)", "Fungi | Candida albicans", "Fungi | Aspergillus fumigatus", "Fungi | Cryptococcus neoformans", "Helminth | Schistosoma mansoni", "Helminth | Brugia malayi (filariasis)", "Human | Homo sapiens (human disease targets / off-target selectivity)", "Custom | type the organism / taxon name in CUSTOM_ORGANISM"]
CUSTOM_ORGANISM = "" #@param {type:"string"}
#@markdown *(only used if ORGANISM = Custom: any taxon name as in the NCBI taxonomy, e.g. `Trypanosoma`, `Candida auris`, `Middle East respiratory syndrome-related coronavirus`)*
TARGET_KEYWORDS = "" #@param {type:"string"}
#@markdown *(optional, comma-separated words to keep only some proteins, e.g. `dihydrofolate, gyrase, protease`)*
MAX_RESOLUTION = 2.5 #@param {type:"number"}
MAX_TARGETS = 20 #@param {type:"integer"}
#@markdown ### Ligand library
LIBRARY_SOURCE = "Upload files (.smi .csv .tsv .sdf .sdf.gz .mol2 .xlsx)" #@param ["Upload files (.smi .csv .tsv .sdf .sdf.gz .mol2 .xlsx)", "Paste SMILES (cell below)", "ChEMBL approved drugs (API)", "ChEMBL ligands of a target (CHEMBL_TARGET_ID)", "PubChem similarity search (QUERY_SMILES)", "PubChem compound names (PUBCHEM_NAMES)", "Download from URL (LIB_URL)", "Demo: 3 CF3-thiazole hydrazones"]
LIB_PATH = "" #@param {type:"string"}
#@markdown *(extra files already on disk / Google Drive, comma-separated paths or wildcards, e.g. `/content/drive/MyDrive/libs/*.sdf`)*
CHEMBL_TARGET_ID = "" #@param {type:"string"}
QUERY_SMILES = "" #@param {type:"string"}
PUBCHEM_NAMES = "" #@param {type:"string"}
LIB_URL = "" #@param {type:"string"}
LIB_MAX_MOLECULES = 200 #@param {type:"integer"}
LIB_PRESCREEN = "similarity to native ligands" #@param ["similarity to native ligands", "diverse subset", "none (first N)"]
MIN_MW = 150 #@param {type:"number"}
MAX_MW = 600 #@param {type:"number"}
#@markdown ### Cut-offs and modules
REL_CUTOFF = 0.8 #@param {type:"number"}
CNN_SCORE_CUTOFF = 0.5 #@param {type:"number"}
SAR_MODE = "auto (only if the library contains the CF3-thiazole parents)" #@param ["auto (only if the library contains the CF3-thiazole parents)", "on", "off"]
LIB_PASTE = ""

ORG_CATALOG = {'Protozoa | Leishmania (leishmaniasis)': ('Leishmania', 'protozoa'), 'Protozoa | Trypanosoma cruzi (Chagas disease)': ('Trypanosoma cruzi', 'protozoa'), 'Protozoa | Trypanosoma brucei (sleeping sickness)': ('Trypanosoma brucei', 'protozoa'), 'Protozoa | Plasmodium falciparum (malaria)': ('Plasmodium falciparum', 'protozoa'), 'Protozoa | Plasmodium vivax (malaria)': ('Plasmodium vivax', 'protozoa'), 'Protozoa | Plasmodium - any species': ('Plasmodium', 'protozoa'), 'Protozoa | Toxoplasma gondii': ('Toxoplasma gondii', 'protozoa'), 'Protozoa | Cryptosporidium': ('Cryptosporidium', 'protozoa'), 'Protozoa | Giardia': ('Giardia', 'protozoa'), 'Protozoa | Trichomonas vaginalis': ('Trichomonas vaginalis', 'protozoa'), 'Amoeba | Entamoeba histolytica (amoebiasis)': ('Entamoeba histolytica', 'protozoa'), 'Amoeba | Naegleria fowleri (brain-eating amoeba)': ('Naegleria fowleri', 'protozoa'), 'Amoeba | Acanthamoeba': ('Acanthamoeba', 'protozoa'), 'Bacteria | Mycobacterium tuberculosis': ('Mycobacterium tuberculosis', 'bacteria'), 'Bacteria | Mycobacterium - any species': ('Mycobacterium', 'bacteria'), 'Bacteria | Staphylococcus aureus (incl. MRSA)': ('Staphylococcus aureus', 'bacteria'), 'Bacteria | Escherichia coli': ('Escherichia coli', 'bacteria'), 'Bacteria | Pseudomonas aeruginosa': ('Pseudomonas aeruginosa', 'bacteria'), 'Bacteria | Klebsiella pneumoniae': ('Klebsiella pneumoniae', 'bacteria'), 'Bacteria | Acinetobacter baumannii': ('Acinetobacter baumannii', 'bacteria'), 'Bacteria | Helicobacter pylori': ('Helicobacter pylori', 'bacteria'), 'Bacteria | Streptococcus pneumoniae': ('Streptococcus pneumoniae', 'bacteria'), 'Bacteria | Enterococcus faecalis': ('Enterococcus faecalis', 'bacteria'), 'Bacteria | Salmonella': ('Salmonella', 'bacteria'), 'Bacteria | Vibrio cholerae': ('Vibrio cholerae', 'bacteria'), 'Bacteria | Clostridioides difficile': ('Clostridioides difficile', 'bacteria'), 'Bacteria | Neisseria gonorrhoeae': ('Neisseria gonorrhoeae', 'bacteria'), 'Bacteria | Bacillus anthracis': ('Bacillus anthracis', 'bacteria'), 'Bacteria | Burkholderia pseudomallei': ('Burkholderia pseudomallei', 'bacteria'), 'Bacteria | ALL bacteria (very large - raise MAX_TARGETS / use keywords)': ('Bacteria', 'bacteria'), 'Virus | SARS-CoV-2': ('Severe acute respiratory syndrome coronavirus 2', 'virus'), 'Virus | HIV-1': ('Human immunodeficiency virus 1', 'virus'), 'Virus | Hepatitis C virus': ('Hepatitis C virus', 'virus'), 'Virus | Hepatitis B virus': ('Hepatitis B virus', 'virus'), 'Virus | Influenza A virus': ('Influenza A virus', 'virus'), 'Virus | Dengue virus': ('Dengue virus', 'virus'), 'Virus | Zika virus': ('Zika virus', 'virus'), 'Virus | Ebola virus (Zaire)': ('Zaire ebolavirus', 'virus'), 'Virus | Chikungunya virus': ('Chikungunya virus', 'virus'), 'Virus | Herpes simplex virus 1': ('Human alphaherpesvirus 1', 'virus'), 'Virus | ALL viruses (very large - use keywords)': ('Viruses', 'virus'), 'Fungi | Candida albicans': ('Candida albicans', 'fungi'), 'Fungi | Aspergillus fumigatus': ('Aspergillus fumigatus', 'fungi'), 'Fungi | Cryptococcus neoformans': ('Cryptococcus neoformans', 'fungi'), 'Helminth | Schistosoma mansoni': ('Schistosoma mansoni', 'protozoa'), 'Helminth | Brugia malayi (filariasis)': ('Brugia malayi', 'protozoa'), 'Human | Homo sapiens (human disease targets / off-target selectivity)': ('Homo sapiens', 'human'), 'Custom | type the organism / taxon name in CUSTOM_ORGANISM': ('', 'protozoa')}
PRIORITY_BY_GROUP = {'protozoa': ['pteridine reductase', 'dihydrofolate', 'trypanothione', 'myristoyl', '14-alpha', 'sterol', 'cytochrome p450', 'kinase', 'cysteine', 'cathepsin', 'falcipain', 'peptidase', 'protease', 'topoisomerase', 'arginase', 'glyceraldehyde', 'hexokinase', 'phosphodiesterase', 'dihydroorotate', 'methyltransferase', 'synthase', 'reductase', 'dehydrogenase'], 'bacteria': ['gyrase', 'topoisomerase', 'dihydrofolate', 'dihydropteroate', 'penicillin-binding', 'beta-lactamase', 'enoyl-', 'inha', 'fabi', 'leucyl', 'tyrosyl', 'aminoacyl', 'peptide deformylase', 'lpxc', 'mura', 'mur ', 'ligase', 'thymidylate', 'dihydroorotate', 'decaprenyl', 'cyp121', 'sortase', 'kinase', 'reductase', 'synthase'], 'virus': ['3c-like', 'main protease', 'protease', 'reverse transcriptase', 'integrase', 'polymerase', 'neuraminidase', 'helicase', 'ns3', 'ns5', 'endonuclease', 'capsid', 'nucleoprotein', 'hemagglutinin', 'spike'], 'fungi': ['14-alpha', 'lanosterol', 'sterol', 'glucan synthase', 'chitin', 'dihydrofolate', 'thymidylate', 'kinase', 'reductase', 'synthase'], 'human': ['kinase', 'protease', 'receptor', 'reductase', 'synthase', 'dehydrogenase']}
import re
ORG_TERM, ORG_GROUP = ORG_CATALOG[ORGANISM]
if CUSTOM_ORGANISM.strip(): ORG_TERM = CUSTOM_ORGANISM.strip()
assert ORG_TERM, 'Choose an organism, or type a taxon name in CUSTOM_ORGANISM'
ORG_MATCH = re.sub(r'\(.*\)','',ORG_TERM).strip().lower()
TARGET_KW = [k.strip().lower() for k in TARGET_KEYWORDS.split(',') if k.strip()]
print('target organism:',ORG_TERM,'| class:',ORG_GROUP,'| library source:',LIBRARY_SOURCE)

N_SHORTLIST = 20; SHORTLIST_DRUGLIKE = True; SHORTLIST_PB = True; VIEW_TOP = 0
POSE_CUTOFF = dict(rel_min=REL_CUTOFF, cnn_score_min=CNN_SCORE_CUTOFF, cnn_aff_min=None, min_aff_max=None, top_per_target=25, max_pairs=300, min_hbond=0, run_posebusters=True)

**Paste SMILES here** (one per line, optional name after a space) - only used when *LIBRARY_SOURCE = Paste SMILES*. Run this cell after cell 0.

In [ ]:
LIB_PASTE = """
# SMILES name
FC(F)(F)c1ccccc1C=NNc1nc(-c2ccc(C#N)cc2)cs1  example_1
CC(=O)Oc1ccccc1C(=O)O  aspirin
"""

In [ ]:
import os, re, json, sys, time, math, shutil, subprocess, itertools, warnings
from pathlib import Path
import numpy as np, pandas as pd, requests
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
from rdkit import Chem, RDLogger, RDConfig
from rdkit.Chem import AllChem, Descriptors, Crippen, rdMolDescriptors, QED, Draw, rdMolAlign
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
RDLogger.DisableLog('rdApp.*')

WORK = Path('/content/screening_work'); (WORK/'dock').mkdir(parents=True, exist_ok=True); os.chdir(WORK)

CFG = dict(
    max_targets=MAX_TARGETS,        # max number of distinct Leishmania proteins to screen
    # ---- target quality gates (a protein must pass ALL of them to be a "validated target") ----
    max_resolution=MAX_RESOLUTION,    # A   crystal resolution
    max_rfree=0.27,        #     R-free (skipped only if the entry reports none)
    min_rscc=0.85,         #     real-space correlation of the co-crystal ligand with the density (ligand really is there)
    min_lig_mw=200.0, max_lig_mw=650.0, min_lig_heavy=14,   # drug-like co-crystal ligand
    min_contacts=45,       #     protein heavy atoms within 4.5 A of the ligand (= a buried, real binding pocket)
    redock_rmsd_ok=2.0,    # A   re-docking must reproduce the crystal pose  -> "validated"
    redock_rmsd_prov=3.0,  # A   2-3 A -> "provisional" (used only if fewer than min_validated targets are validated)
    min_validated=3,
    require_posebusters=False,   # True = a target is only 'validated' if its re-docked pose also passes PoseBusters (stricter)
    site_size=24.0,        # A   edge of the active-site box used for targets without a drug-like ligand
    exhaustiveness=8, num_modes=9, autobox_add=6, seed=0,
    n_gen2=300,            # gen-2 molecules to dock
    n_top_targets=5,       # gen-2 is docked on the best N targets from gen-1
)
INCLUDE_SITE_ONLY = False   # True = also dock proteins that have NO drug-like co-crystal ligand, in a box around the active site (exploratory, flagged, never "validated")
# Manual active-site box for any PDB (overrides everything). Give a centre OR residues (chain, resseq):
#   MANUAL_SITES = {'1E7W': dict(residues=[('A',13),('A',161),('A',164)]), '3L4D': dict(center=(10.0,22.5,-3.1))}
MANUAL_SITES   = {}
EXCLUDE_NAME_WORDS = ['GLUTATHIONE','PEPTIDE','NUCLEOTIDE','SACCHARIDE','GLUCOSE','URIDINE','ADENOSINE','GUANOSINE','COENZYME','HEME']  # substrates / cofactors are not "drug-like ligands"
MANUAL_PDBS    = []   # force-include, e.g. ['1E7W'] (PTR1) - still has to pass the quality gates
OFFTARGET_PDBS = []   # optional human orthologs (e.g. human DHFR) for a selectivity estimate; same prep applies
CFG['use_provisional'] = False
for _n in ('targets','site_targets','targets_all','site_targets_all','USE_T','USE_T_all','LIB','LIB_ALL','res_lib','R_ALL','final','val','smiles_map','SAR_ENUM','ADME_T'): globals().setdefault(_n, None)

In [ ]:
#@title UI toolkit - helpers shared by every form (no need to edit) { display-mode: "form" }
import gradio as gr, threading, io, sys, time, html as _html, contextlib, traceback
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

AUTO = MODE.startswith('run')              # True = no forms, every stage runs with the settings of cell 0
try:
    import google.colab                    # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if not AUTO:
    # Forms draw their figures off-screen in worker threads and collect them as images. Gradio normally flips matplotlib's
    # backend around every event, which can collide with a running stage - so fix the backend once and switch that off.
    try: matplotlib.use('Agg')
    except Exception: pass
    try:
        import gradio.utils as _gu
        class _KeepBackend:
            def __enter__(self): return self
            def __exit__(self, *a): return False
        for _n in ('MatplotlibBackendMananger', 'MatplotlibBackendManager'):
            if hasattr(_gu, _n): setattr(_gu, _n, _KeepBackend)
    except Exception: pass

UI = dict(servers={}, stop=False, lock=threading.Lock())
UPLOADED_FILES = []                        # files chosen in the Library form
LIB_SOURCES = ["Upload files (.smi .csv .tsv .sdf .sdf.gz .mol2 .xlsx)", "Paste SMILES (cell below)", "ChEMBL approved drugs (API)",
               "ChEMBL ligands of a target (CHEMBL_TARGET_ID)", "PubChem similarity search (QUERY_SMILES)",
               "PubChem compound names (PUBCHEM_NAMES)", "Download from URL (LIB_URL)", "Demo: 3 CF3-thiazole hydrazones"]
class StopRequested(Exception):
    pass


# ------------------------------------------------------------------ capture of a stage's output
class Capture:
    """Collects what a stage prints / displays / plots so a Gradio form can show it."""
    def __init__(self):
        self.parts = []; self.tables = []; self.images = []; self.html3d = []; self._lock = threading.Lock(); self._tq = {}
    def write(self, s):
        with self._lock: self.parts.append(s)
        return len(s)
    def flush(self): pass
    def isatty(self): return False
    def text(self, last=12000):
        with self._lock: t = ''.join(self.parts)
        return t[-last:]
    def display(self, *objs, **k):
        for o in objs: self._one(o)
    def _one(self, o):
        import pandas as pd
        if isinstance(o, pd.DataFrame): self.tables.append(o.copy())
        elif isinstance(o, pd.Series): self.tables.append(o.to_frame())
        elif hasattr(o, 'data') and hasattr(o, '_repr_png_') and isinstance(getattr(o, 'data', None), (bytes, bytearray)):
            self.images.append(Image.open(io.BytesIO(o.data)).copy())
        elif isinstance(o, Image.Image): self.images.append(o)
        elif hasattr(o, 'to_html') and hasattr(o, 'data'):                       # pandas Styler
            self.tables.append(o.data.copy())
        else: print(o)
    def plt_show(self, *a, **k):
        for n in plt.get_fignums():
            fig = plt.figure(n); buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=110, bbox_inches='tight'); buf.seek(0)
            self.images.append(Image.open(buf).copy()); plt.close(fig)
    def tqdm(self, it, total=None, desc='', **k):
        try: total = total or len(it)
        except Exception: total = None
        step = 1 if (total or 0) <= 60 else max(1, total // 40); t0 = time.time()
        for i, x in enumerate(it):
            if UI['stop']: raise StopRequested()
            if i % step == 0 or i + 1 == total: print(f"[{desc or 'progress'}] {i + 1}/{total or '?'}   {time.time() - t0:.0f} s")
            yield x


@contextlib.contextmanager
def capturing(cap):
    """Redirect print / display / plt.show / py3Dmol .show / tqdm of the notebook functions into `cap`."""
    import py3Dmol
    g = globals(); _MISSING = object()
    old = {k: g.get(k, _MISSING) for k in ('display', 'tqdm')}
    g['display'] = cap.display; g['tqdm'] = cap.tqdm
    old_show = plt.show; plt.show = cap.plt_show
    old_view = py3Dmol.view.show; py3Dmol.view.show = lambda self, *a, **k: cap.html3d.append(self._make_html())
    old_out, old_err = sys.stdout, sys.stderr; sys.stdout = cap; sys.stderr = cap
    try:
        yield cap
    finally:
        sys.stdout, sys.stderr = old_out, old_err
        plt.show = old_show; py3Dmol.view.show = old_view
        for k, v in old.items():
            if v is _MISSING: g.pop(k, None)
            else: g[k] = v


def stream_job(fn, *a, **k):
    """Run fn in a worker thread under capture. Generator of (cap, state, result) about once a second;
    state is 'running' and finally 'done' | 'error' | 'stopped'."""
    cap = Capture(); res = {}
    if not UI['lock'].acquire(blocking=False):
        cap.write('Another stage is still running - wait for it to finish (or press Stop).\n'); yield cap, 'error', res; return
    UI['stop'] = False
    def work():
        try:
            with capturing(cap): res['value'] = fn(*a, **k)
        except StopRequested: cap.write('\n[stopped by user]\n'); res['stopped'] = True
        except Exception: cap.write('\n' + traceback.format_exc()); res['error'] = True
        finally: UI['lock'].release()
    th = threading.Thread(target=work, daemon=True); th.start()
    while th.is_alive():
        yield cap, 'running', res; time.sleep(1.0)
    th.join()
    yield cap, ('error' if res.get('error') else 'stopped' if res.get('stopped') else 'done'), res


def need(*names):
    miss = [n for n in names if globals().get(n) is None]
    if miss: raise RuntimeError('Run the previous stage first - missing: ' + ', '.join(miss))


# ------------------------------------------------------------------ rendering helpers
def show_df(df, n=1000, drop=('molblock',)):
    import pandas as pd, numpy as np
    if df is None: return None
    d = df.drop(columns=[c for c in drop if c in df.columns]).head(n).copy()
    for c in d.columns:
        if d[c].dtype == object: d[c] = d[c].map(lambda v: v if isinstance(v, (str, int, float, bool, type(None))) else str(v))
        elif pd.api.types.is_float_dtype(d[c]): d[c] = d[c].round(3)
    return d.reset_index() if (d.index.name or not isinstance(d.index, pd.RangeIndex)) else d


def fig_to_pil(fig, dpi=110):
    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight'); buf.seek(0); plt.close(fig)
    return Image.open(buf).copy()


def iframe_html(page, height=560):
    return f'<iframe style="width:100%;height:{height}px;border:1px solid #ddd;border-radius:6px" srcdoc="{_html.escape(page, quote=True)}"></iframe>'


def mol_grid(smiles, legends=None, n=24, per_row=6, size=(200, 170)):
    mols = [Chem.MolFromSmiles(s) for s in list(smiles)[:n]]
    return Draw.MolsToGridImage(mols, molsPerRow=per_row, subImgSize=size, legends=[str(x) for x in (list(legends) if legends is not None else [''] * len(mols))][:len(mols)])


# ------------------------------------------------------------------ settings written by the forms
def set_target_settings(organism, custom, keywords, max_res, max_rfree, min_rscc, min_contacts, max_targets):
    global ORGANISM, CUSTOM_ORGANISM, TARGET_KEYWORDS, MAX_RESOLUTION, MAX_TARGETS, ORG_TERM, ORG_GROUP, ORG_MATCH, TARGET_KW
    ORGANISM, CUSTOM_ORGANISM, TARGET_KEYWORDS = organism, custom, keywords
    MAX_RESOLUTION, MAX_TARGETS = float(max_res), int(max_targets)
    ORG_TERM, ORG_GROUP = ORG_CATALOG[ORGANISM]
    if (custom or '').strip(): ORG_TERM = custom.strip()
    if not ORG_TERM: raise ValueError('Type a taxon name in the Custom box')
    ORG_MATCH = re.sub(r'\(.*\)', '', ORG_TERM).strip().lower()
    TARGET_KW = [k.strip().lower() for k in (keywords or '').split(',') if k.strip()]
    CFG.update(max_targets=MAX_TARGETS, max_resolution=MAX_RESOLUTION, max_rfree=float(max_rfree), min_rscc=float(min_rscc), min_contacts=int(min_contacts))


def set_library_settings(source, paste, chembl_id, query_smiles, pubchem_names, url, min_mw, max_mw, max_mols, prescreen_mode):
    global LIBRARY_SOURCE, LIB_PASTE, CHEMBL_TARGET_ID, QUERY_SMILES, PUBCHEM_NAMES, LIB_URL, MIN_MW, MAX_MW, LIB_MAX_MOLECULES, LIB_PRESCREEN, LIB_PATH
    LIBRARY_SOURCE, LIB_PASTE, CHEMBL_TARGET_ID, QUERY_SMILES, PUBCHEM_NAMES, LIB_URL = source, paste, chembl_id, query_smiles, pubchem_names, url
    MIN_MW, MAX_MW, LIB_MAX_MOLECULES, LIB_PRESCREEN, LIB_PATH = float(min_mw), float(max_mw), int(max_mols), prescreen_mode, ''


# ------------------------------------------------------------------ launching
def launch_form(name, demo, height=820):
    """Launch one Gradio form below the cell. Re-running a form cell closes the previous copy of that form."""
    old = UI['servers'].pop(name, None)
    if old is not None:
        try: old.close()
        except Exception: pass
    kw = dict(prevent_thread_lock=True, quiet=True, inline=True, height=height)
    pw = (UI_PASSWORD or '').strip()
    if pw: kw['auth'] = ('screening', pw)
    if IN_COLAB: kw['share'] = True        # Gradio cannot serve inside Colab without its tunnel
    demo.queue(default_concurrency_limit=4)
    demo.launch(**kw)
    UI['servers'][name] = demo
    if IN_COLAB and not pw and not UI.get('warned'):
        UI['warned'] = True
        print('NOTE: in Colab every form is reachable through a public gradio.live link. Set UI_PASSWORD in cell 0 (login: "screening") if you share this session.')
    return demo


def run_or_launch(name, builder, stage_fn=None):
    """interactive mode -> open the form; 'run everything' mode -> run the stage directly with the settings of cell 0"""
    if AUTO:
        if stage_fn is not None: stage_fn()
    else:
        launch_form(name, builder())

## 1. GNINA (GPU build if possible, CPU fallback)

In [ ]:
GNINA = str(WORK/'gnina'); USE_GPU = shutil.which('nvidia-smi') is not None
def _gnina_ok():
    r = subprocess.run(f"{GNINA} --version", shell=True, capture_output=True, text=True)
    return r.returncode == 0
def install_gnina():
    if Path(GNINA).exists() and _gnina_ok(): return
    urls = []
    try:
        rel = requests.get('https://api.github.com/repos/gnina/gnina/releases/latest', timeout=30).json()
        for a in rel.get('assets', []):
            if re.fullmatch(r'gnina(\.[\d.]+)?', a['name']): urls.append(a['browser_download_url'])
    except Exception as e:
        print('release lookup failed:', e)
    urls.append('https://github.com/gnina/gnina/releases/download/v1.0.3/gnina')   # known-good fallback (CPU)
    for u in urls:
        sh(f"wget -q -O {GNINA} {u} && chmod +x {GNINA}", check=False)
        if _gnina_ok():
            print('GNINA from', u); return
    raise RuntimeError('could not install gnina')
install_gnina()
print(sh(f"{GNINA} --version", check=False).stdout.splitlines()[:1], '| GPU:', USE_GPU)

## Step 1 - Targets

In [ ]:
SOLVENT = {'HOH','DOD','GOL','EDO','PEG','PGE','PG4','P6G','1PE','SO4','PO4','ACT','ACY','DMS','CL','NA','K','BR','IOD','FMT','TRS','MES','EPE','BME','IMD','CIT','TLA','MPD','DTT','NO3','SCN','EOH','IPA','PGO','NH4','CO3','OXY','AZI'}
COFACTORS = {'NDP','NAP','NAD','NAI','NDA','FAD','FMN','HEM','HEC','SAM','SAH','ATP','ADP','ANP','ACP','GTP','GDP','TPP','PLP','COA'}   # organic cofactors kept in the receptor
METALS_MODS = {'ZN','MG','MN','FE','FE2','CU','NI','CO','CA','MSE','SEP','TPO','PTR','CSO','KCX','CME'}
KEEP_HET = COFACTORS | METALS_MODS
NONLIG = SOLVENT | KEEP_HET | {'GSH','UPG','UDP','UD1','GDD','NGA','NAG','MAN','BMA','GAL','FUC'}
PRIORITY = PRIORITY_BY_GROUP.get(ORG_GROUP,[])+PRIORITY_BY_GROUP['protozoa']   # target classes of this organism group first

SEARCH='https://search.rcsb.org/rcsbsearch/v2/query'; GQL='https://data.rcsb.org/graphql'
def rcsb_ids(max_res):
    def t(attr, op, val): return {"type":"terminal","service":"text","parameters":{"attribute":attr,"operator":op,"value":val}}
    q={"query":{"type":"group","logical_operator":"and","nodes":[
        t("rcsb_entity_source_organism.taxonomy_lineage.name","exact_match",ORG_TERM),
        t("rcsb_entry_info.resolution_combined","less_or_equal",max_res),
        t("rcsb_entry_info.nonpolymer_entity_count","greater",0),
        t("exptl.method","exact_match","X-RAY DIFFRACTION")]},
       "return_type":"entry","request_options":{"paginate":{"start":0,"rows":10000}}}
    r=requests.post(SEARCH,json=q,timeout=120); r.raise_for_status()
    return [x['identifier'] for x in r.json().get('result_set',[])]

_POLY = """polymer_entities{rcsb_polymer_entity{pdbx_description} rcsb_entity_source_organism{ncbi_scientific_name}
 rcsb_polymer_entity_container_identifiers{reference_sequence_identifiers{database_name database_accession}}}"""
GQL_FULL = "query($ids:[String!]!){entries(entry_ids:$ids){rcsb_id struct{title} rcsb_entry_info{resolution_combined} refine{ls_R_factor_R_free} " + _POLY + \
  " nonpolymer_entities{nonpolymer_comp{chem_comp{id formula_weight name formula}} nonpolymer_entity_instances{rcsb_nonpolymer_entity_instance_container_identifiers{comp_id auth_asym_id} rcsb_nonpolymer_instance_validation_score{RSCC RSR is_best_instance}}}}}"
GQL_LEAN = "query($ids:[String!]!){entries(entry_ids:$ids){rcsb_id struct{title} rcsb_entry_info{resolution_combined} " + _POLY + \
  " nonpolymer_entities{nonpolymer_comp{chem_comp{id formula_weight name formula}}}}}"
QC_OK = True
def fetch_meta(ids):
    global QC_OK
    out=[]
    for i in range(0,len(ids),100):
        chunk=ids[i:i+100]; r=None
        for q in ([GQL_FULL,GQL_LEAN] if QC_OK else [GQL_LEAN]):
            r=requests.post(GQL,json={'query':q,'variables':{'ids':chunk}},timeout=120).json()
            if r.get('data') and r['data'].get('entries') is not None and not r.get('errors'): break
            if q is GQL_FULL:
                QC_OK=False; print('  [warning] RCSB ligand-validation fields unavailable -> RSCC/R-free gates cannot be applied; falling back to a lean query')
        out+= [e for e in (r['data']['entries'] if r and r.get('data') else []) if e]
    return out

def _rscc(inst_list, comp):
    v=[]
    for ins in inst_list or []:
        cid=(ins.get('rcsb_nonpolymer_entity_instance_container_identifiers') or {}).get('comp_id')
        for sc in ins.get('rcsb_nonpolymer_instance_validation_score') or []:
            if sc and sc.get('RSCC') is not None and (cid in (None,comp)): v.append(sc['RSCC'])
    return max(v) if v else None

def summarize(e):
    pe=e.get('polymer_entities') or []
    _o=lambda p:' '.join(o['ncbi_scientific_name'] for o in (p.get('rcsb_entity_source_organism') or []) if o.get('ncbi_scientific_name')).lower()
    pe_m=[p for p in pe if ORG_MATCH in _o(p)] or pe      # describe the polymer chain that really comes from the target organism (not an antibody / host partner)
    orgs=sorted({o['ncbi_scientific_name'] for p in pe for o in (p.get('rcsb_entity_source_organism') or []) if o.get('ncbi_scientific_name')})
    desc=[(p.get('rcsb_polymer_entity') or {}).get('pdbx_description','') for p in pe_m]
    unis=[ri['database_accession'] for p in pe_m for ri in ((p.get('rcsb_polymer_entity_container_identifiers') or {}).get('reference_sequence_identifiers') or []) if ri['database_name']=='UniProt']
    rf=[x.get('ls_R_factor_R_free') for x in (e.get('refine') or []) if x and x.get('ls_R_factor_R_free')]
    rfree=min(rf) if rf else None
    ligs=[]; qc={}; cofs=[]
    for n in (e.get('nonpolymer_entities') or []):
        cc=n['nonpolymer_comp']['chem_comp']; cid=cc['id']; w=cc.get('formula_weight'); nm=(cc.get('name') or '').upper(); fo=cc.get('formula') or ''
        if cid in COFACTORS: cofs.append(cid)
        if cid in NONLIG or not w or not (CFG['min_lig_mw']<=w<=CFG['max_lig_mw']): continue
        if re.search(r'\bP\d*\b',fo) or any(k in nm for k in EXCLUDE_NAME_WORDS): continue   # phosphates/nucleotides/substrates are not drug-like ligands
        rs=_rscc(n.get('nonpolymer_entity_instances'),cid)
        if QC_OK and (rs is None or rs<CFG['min_rscc']): continue      # ligand not convincingly supported by the density
        ligs.append(cid); qc[cid]=rs
    res=((e.get('rcsb_entry_info') or {}).get('resolution_combined') or [9.9])[0]
    prot=desc[0] if desc else ''
    return dict(pdb=e['rcsb_id'],title=e['struct']['title'],protein=prot,organism='; '.join(orgs),uniprot=(unis[0] if unis else prot.lower()),
                res=res,rfree=rfree,ligs=ligs,lig_rscc=qc,cofs=sorted(set(cofs)))
def prio(p):
    p=p.lower()
    for i,k in enumerate(PRIORITY):
        if k in p: return i
    return len(PRIORITY)

def stage_targets():
    global PRIORITY, _m, best, best_per_protein, cand, ex, extra, ids, nolig, off, site_targets, site_targets_all, targets, targets_all
    PRIORITY = PRIORITY_BY_GROUP.get(ORG_GROUP,[])+PRIORITY_BY_GROUP["protozoa"]
    ids = rcsb_ids(CFG['max_resolution'])
    print(len(ids),f"{ORG_TERM} X-ray entries with resolution <= {CFG['max_resolution']} A and a non-polymer ligand")
    cand=pd.DataFrame([summarize(e) for e in fetch_meta(ids)])
    _m=cand.organism.str.contains(re.escape(ORG_MATCH),case=False)
    if _m.sum()==0: print('  [note] organism-name check skipped (RCSB lineage name differs from the names used inside the entries)')
    else: cand=cand[_m]
    cand=cand[cand.rfree.isna()|(cand.rfree<=CFG['max_rfree'])]
    if TARGET_KW: cand=cand[cand.protein.str.lower().map(lambda p:any(k in p for k in TARGET_KW))]; print('keyword filter',TARGET_KW,'->',len(cand),'entries')
    print(len(cand),'pass resolution + R-free;',(cand.ligs.str.len()>0).sum(),'of them carry a drug-like, density-supported ligand')

    def best_per_protein(df):
        df=df.assign(best_rscc=df.lig_rscc.map(lambda d: max([v for v in d.values() if v is not None],default=0)))
        return df.sort_values(['res','best_rscc'],ascending=[True,False]).groupby('uniprot',as_index=False).first()

    best=best_per_protein(cand[cand.ligs.str.len()>0])
    best['prio']=best.protein.map(prio); best['role']='target'; best['tier_in']='ligand'
    targets=best.sort_values(['prio','res']).head(CFG['max_targets']).reset_index(drop=True)

    extra=[i for i in MANUAL_PDBS if i not in set(targets.pdb)]
    if extra:
        ex=pd.DataFrame([summarize(e) for e in fetch_meta(extra)]); ex=ex[ex.ligs.str.len()>0]
        ex['role']='target'; ex['prio']=0; ex['tier_in']='ligand'
        targets=pd.concat([ex,targets],ignore_index=True)
    if OFFTARGET_PDBS:
        off=pd.DataFrame([summarize(e) for e in fetch_meta(OFFTARGET_PDBS)]); off['role']='off'; off['prio']=99; off['tier_in']='ligand'
        targets=pd.concat([targets,off],ignore_index=True)
    targets=targets[targets.ligs.str.len()>0].reset_index(drop=True)

    # proteins with NO drug-like co-crystal ligand but a cofactor/catalytic site -> "active-site-only" candidates
    nolig=cand[(cand.ligs.str.len()==0)&(cand.cofs.str.len()>0)&(~cand.uniprot.isin(targets.uniprot))]
    site_targets=best_per_protein(nolig) if len(nolig) else nolig.copy()
    if len(site_targets):
        site_targets['prio']=site_targets.protein.map(prio); site_targets['role']='target'; site_targets['tier_in']='site'
        site_targets=site_targets.sort_values(['prio','res']).head(CFG['max_targets']).reset_index(drop=True)

    pd.set_option('display.max_colwidth',60,'display.width',200)
    print('\nTARGETS passing all gates (drug-like co-crystal ligand):')
    display(targets[['pdb','protein','organism','res','rfree','ligs','lig_rscc','role']])
    if len(site_targets):
        print('\nACTIVE-SITE-ONLY candidates (no drug-like ligand; docked only if INCLUDE_SITE_ONLY=True):')
        display(site_targets[['pdb','protein','res','rfree','cofs']])
    targets_all = targets.copy(); site_targets_all = site_targets.copy()

In [ ]:
#@title Step 1 - FORM: target organism and protein-quality gates { display-mode: "form" }
def build_target_form():
    groups = list(dict.fromkeys(k.split(' | ')[0] for k in ORG_CATALOG))
    orgs_of = lambda g: [k for k in ORG_CATALOG if k.startswith(g + ' | ')]
    g0 = ORGANISM.split(' | ')[0]
    with gr.Blocks(title='Step 1 - Targets') as demo:
        gr.Markdown('## Step 1 - Target organism and protein-quality gates\nPick the organism, set the quality gates, retrieve its proteins from the RCSB PDB, then tick the ones to keep.')
        with gr.Row():
            grp = gr.Radio(groups, value=g0, label='Organism group')
            org = gr.Dropdown(orgs_of(g0), value=ORGANISM, label='Organism')
        custom = gr.Textbox(value=CUSTOM_ORGANISM, label='Custom taxon (used when the group is "Custom") - any NCBI taxon name, e.g. Trypanosoma, Candida auris')
        kw = gr.Textbox(value=TARGET_KEYWORDS, label='Keep only proteins whose name contains (comma-separated, optional), e.g. dihydrofolate, gyrase, protease')
        with gr.Row():
            res = gr.Slider(1.0, 3.5, value=MAX_RESOLUTION, step=0.1, label='Max resolution (Å)')
            rfree = gr.Slider(0.15, 0.35, value=CFG['max_rfree'], step=0.01, label='Max R-free')
            rscc = gr.Slider(0.5, 1.0, value=CFG['min_rscc'], step=0.01, label='Min ligand density fit (RSCC)')
        with gr.Row():
            cont = gr.Slider(10, 100, value=CFG['min_contacts'], step=1, label='Min pocket contacts (protein atoms within 4.5 Å of the ligand)')
            maxt = gr.Slider(1, 100, value=MAX_TARGETS, step=1, label='Max number of proteins')
        run = gr.Button('Retrieve targets from the RCSB PDB', variant='primary')
        msg = gr.Markdown()
        log = gr.Textbox(label='Log', lines=7, interactive=False)
        t_ok = gr.Dataframe(label='Proteins that pass all gates (drug-like, density-supported co-crystal ligand)', interactive=False, wrap=True)
        t_site = gr.Dataframe(label='Active-site-only candidates (no drug-like ligand; used only if enabled in step 2)', interactive=False, wrap=True)
        pick = gr.CheckboxGroup(label='Tick the proteins to keep', choices=[], value=[])
        confirm = gr.Button('Keep the ticked proteins')
        out = gr.Markdown()

        grp.change(lambda g: gr.update(choices=orgs_of(g), value=(orgs_of(g) or [None])[0]), grp, org)

        def on_run(o, c, k, r_, rf, rs, ct, mt):
            try: set_target_settings(o, c, k, r_, rf, rs, ct, mt)
            except Exception as e:
                yield str(e), gr.update(), gr.update(), gr.update(), f'**Error:** {e}'; return
            for cap, state, _ in stream_job(stage_targets):
                if state == 'running': yield cap.text(), gr.update(), gr.update(), gr.update(), 'Querying the RCSB (usually 30-90 s) ...'
                elif state == 'done':
                    labels = [(f"{x.pdb} · {str(x.protein)[:45]} · {x.res:.2f} Å", x.pdb) for x in targets.itertuples()]
                    yield (cap.text(), show_df(targets[['pdb', 'protein', 'organism', 'res', 'rfree', 'ligs', 'lig_rscc', 'role']]),
                           show_df(site_targets[['pdb', 'protein', 'res', 'rfree', 'cofs']]) if len(site_targets) else None,
                           gr.update(choices=labels, value=[v for _, v in labels]),
                           f'**{len(targets)} proteins** pass all gates for *{ORG_TERM}*' + (f' (+ {len(site_targets)} active-site-only candidates).' if len(site_targets) else '.') + ' Untick any you do not want, then press *Keep*.' if len(targets) else f'**No protein passed all gates** for *{ORG_TERM}*. Relax the gates (resolution, R-free, RSCC, contacts) or the keyword filter.')
                else: yield cap.text(), gr.update(), gr.update(), gr.update(), f'**{state}** - see the log'
        run.click(on_run, [org, custom, kw, res, rfree, rscc, cont, maxt], [log, t_ok, t_site, pick, msg])

        def on_confirm(ids):
            global targets, site_targets
            need('targets_all')
            targets = targets_all[targets_all.pdb.isin(ids)].reset_index(drop=True)
            site_targets = site_targets_all[site_targets_all.pdb.isin(ids)].reset_index(drop=True)
            return f'**{len(targets)} protein(s) kept** for validation (+ {len(site_targets)} site-only). Next: Step 2.'
        confirm.click(on_confirm, pick, out)
    return demo

run_or_launch('targets', build_target_form, stage_targets)

## Step 2 - Prepare and validate

In [ ]:
def download_pdb(pdb):
    f=WORK/f'{pdb}.pdb'
    if not f.exists():
        r=requests.get(f'https://files.rcsb.org/download/{pdb}.pdb',timeout=120); r.raise_for_status(); f.write_text(r.text)
    return f

def _xyz(l): return (float(l[30:38]),float(l[38:46]),float(l[46:54]))
def _write_receptor(pdb,lines,chain):
    rec=[]
    for l in lines:
        if l[16] not in (' ','A') or l[21]!=chain: continue
        rn=l[17:20].strip()
        if l.startswith('ATOM') or (l.startswith('HETATM') and rn in KEEP_HET): rec.append(l)
    (WORK/f'{pdb}_receptor.pdb').write_text('\n'.join(rec)+'\nEND\n')
    return sorted({l[17:20].strip() for l in rec if l.startswith('HETATM')})

def prepare_structure(pdb, lig_ids):
    """Pick the co-crystal ligand that is truly drug-like AND buried in a real pocket; reject the structure otherwise."""
    from scipy.spatial import cKDTree
    lines=[l for l in download_pdb(pdb).read_text().splitlines() if l.startswith(('ATOM','HETATM'))]
    keep_alt=lambda l: l[16] in (' ','A')
    prot=np.array([_xyz(l) for l in lines if l.startswith('ATOM') and keep_alt(l) and l[76:78].strip()!='H'])
    tree=cKDTree(prot)
    groups={}
    for l in lines:
        if l.startswith('HETATM') and keep_alt(l) and l[17:20].strip() in lig_ids and l[76:78].strip()!='H':
            groups.setdefault((l[17:20].strip(),l[21],l[22:27]),[]).append(l)
    if not groups: raise RuntimeError('no drug-like ligand found in the coordinate file')
    scored=[]
    for key,ll in groups.items():
        xyz=np.array([_xyz(l) for l in ll])
        near=set(i for lst in tree.query_ball_point(xyz,4.5) for i in lst)
        scored.append((len(near),len(ll),key,ll,xyz))
    ok=[x for x in scored if x[1]>=CFG['min_lig_heavy'] and x[0]>=CFG['min_contacts']]
    if not ok:
        b=max(scored); raise RuntimeError(f"ligand {b[2][0]} is not a buried drug-like ligand (heavy atoms={b[1]}, protein contacts={b[0]}; need >= {CFG['min_lig_heavy']} and >= {CFG['min_contacts']})")
    contacts,nheavy,(resn,chain,_),lig_lines,xyz=max(ok,key=lambda x:x[0])
    cofs=_write_receptor(pdb,lines,chain)
    (WORK/f'{pdb}_native.pdb').write_text('\n'.join(lig_lines)+'\nEND\n')
    cen=xyz.mean(0).round(2).tolist()
    near=sorted({(lines_[17:20].strip()+lines_[22:26].strip()) for lines_ in lines if lines_.startswith('ATOM') and keep_alt(lines_) and lines_[21]==chain
                 and np.min(np.linalg.norm(xyz-np.array(_xyz(lines_)),axis=1))<4.0}, key=lambda s:int(re.sub(r'\D','',s) or 0))
    json.dump(dict(mode='autobox',center=cen,ligand=resn),open(WORK/f'{pdb}_box.json','w'))
    return dict(chain=chain,native=resn,cofactors=cofs,contacts=contacts,nheavy=nheavy,center=cen,pocket=' '.join(near))

def prepare_site(pdb):
    """No drug-like ligand: build a box (CFG['site_size'] A) around the active site.
    Priority: MANUAL_SITES  >  organic cofactor centroid (NADPH/FAD/heme/SAM...) of the best chain."""
    lines=[l for l in download_pdb(pdb).read_text().splitlines() if l.startswith(('ATOM','HETATM'))]
    keep_alt=lambda l: l[16] in (' ','A')
    size=CFG['site_size']; chain=None; how=''
    if pdb in MANUAL_SITES:
        m=MANUAL_SITES[pdb]; size=m.get('size',size)
        if 'center' in m: cen=list(m['center']); chain=m.get('chain','A'); how='manual centre'
        else:
            pts=[_xyz(l) for l in lines if l.startswith('ATOM') and l[12:16].strip()=='CA' and any(l[21]==c and int(l[22:26])==r for c,r in m['residues'])]
            if not pts: raise RuntimeError('MANUAL_SITES residues not found')
            cen=np.mean(pts,0).round(2).tolist(); chain=m['residues'][0][0]; how='manual residues'
    else:
        cof=[l for l in lines if l.startswith('HETATM') and keep_alt(l) and l[17:20].strip() in COFACTORS and l[76:78].strip()!='H']
        if not cof: raise RuntimeError('no cofactor and no MANUAL_SITES entry - cannot place an active-site box')
        cnt={}
        for l in cof: cnt[l[21]]=cnt.get(l[21],0)+1
        chain=max(cnt,key=cnt.get); cen=np.mean([_xyz(l) for l in cof if l[21]==chain],0).round(2).tolist(); how='cofactor centroid'
    cofs=_write_receptor(pdb,lines,chain)
    json.dump(dict(mode='box',center=cen,size=size),open(WORK/f'{pdb}_box.json','w'))
    return dict(chain=chain,native='',cofactors=cofs,contacts=np.nan,nheavy=np.nan,center=cen,pocket=how)

def chemcomp_smiles(cid):
    d=requests.get(f'https://data.rcsb.org/rest/v1/core/chemcomp/{cid}',timeout=30).json().get('rcsb_chem_comp_descriptor',{})
    return d.get('SMILES_stereo') or d.get('SMILES') or d.get('smiles_stereo') or d.get('smiles')

In [ ]:
def embed(smi,name):
    m=Chem.MolFromSmiles(smi)
    if m is None: return None
    m=Chem.AddHs(m)
    if AllChem.EmbedMolecule(m,randomSeed=CFG['seed'])!=0:
        if AllChem.EmbedMolecule(m,useRandomCoords=True,randomSeed=CFG['seed'])!=0: return None
    try: AllChem.MMFFOptimizeMolecule(m,maxIters=500)
    except Exception: pass
    m.SetProp('_Name',name); return m

def ikey(m): return Chem.MolToInchiKey(Chem.RemoveHs(m))[:14]

def write_lib(df,path):
    w=Chem.SDWriter(str(path)); k={}
    for r in df.itertuples():
        m=embed(r.smiles,r.id)
        if m is None: print('  [write_lib] could not build 3D for',r.id,'| SMILES =',r.smiles)
        else: w.write(m); k[ikey(m)]=r.id
    w.close(); return k

def run_gnina(rec,lig,ref,out,log,box=None):
    global USE_GPU
    if box and box.get('mode')=='box':      # explicit active-site block (no co-crystal ligand available)
        c=box['center']; z=str(box['size'])
        boxargs=['--center_x',str(c[0]),'--center_y',str(c[1]),'--center_z',str(c[2]),'--size_x',z,'--size_y',z,'--size_z',z]
    else: boxargs=['--autobox_ligand',ref,'--autobox_add',str(CFG['autobox_add'])]
    base=[GNINA,'-r',rec,'-l',lig]+boxargs+[
          '--exhaustiveness',str(CFG['exhaustiveness']),'--num_modes',str(CFG['num_modes']),'--seed',str(CFG['seed']),'-o',out]
    for gpu in ([True,False] if USE_GPU else [False]):
        with open(log,'w') as f: r=subprocess.run(base+([] if gpu else ['--no_gpu']),stdout=f,stderr=subprocess.STDOUT)
        if r.returncode==0 and Path(out).exists() and Path(out).stat().st_size>0:
            if not gpu: USE_GPU=False
            return True
    return False

def read_sdf(path):
    # GNINA output often fails RDKit's strict sanitization (charges/valences after minimisation) and
    # SDMolSupplier then returns None -> every pose silently dropped. Read unsanitized, keep the tags.
    n_bad=0
    for m in Chem.SDMolSupplier(str(path),sanitize=False,removeHs=False):
        if m is None: n_bad+=1; continue
        m.UpdatePropertyCache(strict=False); Chem.FastFindRings(m); yield m
    if n_bad: print('  [read_sdf] unreadable records:',n_bad)

def parse_poses(sdf,key2id,keep_all=False):
    best={}; allp=[]
    for m in read_sdf(sdf):
        if not m.HasProp('CNNscore'): continue
        name=m.GetProp('_Name') if m.HasProp('_Name') else ''
        if name not in key2id.values():
            try: name=key2id.get(ikey(m),name)
            except Exception: pass
        s=float(m.GetProp('CNNscore')); a=float(m.GetProp('CNNaffinity'))
        rec=dict(id=name,minAff=float(m.GetProp('minimizedAffinity')),CNNscore=s,CNNaff=a,CNN_VS=s*a,molblock=Chem.MolToMolBlock(m))
        if keep_all: allp.append((rec,m))
        if name not in best or rec['CNN_VS']>best[name]['CNN_VS']: best[name]=rec
    df=pd.DataFrame(best.values())
    return (df,allp) if keep_all else df

def pb_check(pdb, mol, true_sdf=None):
    """PoseBusters physical-plausibility check of one docked pose (bond lengths/angles, planarity, clashes, protein-ligand overlap...).
    Returns (all_passed | None if PoseBusters unavailable, [names of failed checks])."""
    try:
        from posebusters import PoseBusters
        m=Chem.Mol(mol);
        try: Chem.SanitizeMol(m)
        except Exception: return False,['sanitization (RDKit cannot sanitize this pose)']
        b=PoseBusters(config='redock' if true_sdf else 'dock')
        df=b.bust([m], true_sdf, str(WORK/f'{pdb}_receptor.pdb'))
        row=df.iloc[0]
        failed=[c for c,v in row.items() if v is False or v==False]
        failed=[c for c in failed if not c.startswith('rmsd')]
        return len(failed)==0, failed
    except Exception as e:
        return None,[f'posebusters unavailable: {type(e).__name__}']

def dock_set(pdb,lib,tag):
    out=WORK/'dock'/f'{pdb}_{tag}.sdf'; libf=WORK/'dock'/f'{tag}_lib.sdf'
    if not libf.exists() or not (WORK/'dock'/f'{tag}_keys.json').exists() or libf.stat().st_size==0:
        k=write_lib(lib,libf); (WORK/'dock'/f'{tag}_keys.json').write_text(json.dumps(k))
    k=json.loads((WORK/'dock'/f'{tag}_keys.json').read_text())
    if not k: raise RuntimeError(f'ligand library for {tag} is empty (RDKit could not embed/parse the SMILES: {list(lib.smiles)[:1]})')
    if out.exists() and out.stat().st_size==0: out.unlink()   # stale empty result
    if not out.exists():
        bj=WORK/f'{pdb}_box.json'; box=json.loads(bj.read_text()) if bj.exists() else None
        ok=run_gnina(str(WORK/f'{pdb}_receptor.pdb'),str(libf),str(WORK/f'{pdb}_native.pdb'),str(out),str(WORK/'dock'/f'{pdb}_{tag}.log'),box)
        if not ok:
            print('  docking FAILED',pdb,tag)
            lg=WORK/'dock'/f'{pdb}_{tag}.log'
            if lg.exists(): print('  --- last lines of GNINA log ---\n  '+'\n  '.join(lg.read_text().splitlines()[-15:]))
            return pd.DataFrame()
    d=parse_poses(out,k); d['pdb']=pdb
    if d.empty:
        print('  GNINA ran but no scored poses parsed for',pdb,tag,'| sdf bytes:',out.stat().st_size if out.exists() else 'missing')
        try:
            ms=list(Chem.SDMolSupplier(str(out),sanitize=False)); print('  records:',len(ms),'| first record tags:',list(ms[0].GetPropNames()) if ms and ms[0] else None)
        except Exception as e: print('  could not inspect sdf:',e)
        out.unlink(missing_ok=True)   # do not cache a bad result
    return d

In [ ]:
def stage_validate():
    global P, REJ, S, USE_T, USE_T_all, V, VAL, _tier, cdist, d, e, heavy, lib, m, mm, nat, ok, pb_fail, pb_ok, pdb, poses, prep, prov, r, ref, rmsd_min, rmsd_top, rows, show, smi, t, top, val
    VAL=[]; REJ=[]
    rows=targets.to_dict('records') + (site_targets.to_dict('records') if (INCLUDE_SITE_ONLY and len(site_targets)) else [])
    for t in tqdm(rows):
        pdb=t['pdb']
        try:
            if t.get('tier_in')=='site':
                prep=prepare_site(pdb)
                VAL.append(dict(pdb=pdb,protein=t['protein'],role=t['role'],res=t['res'],chain=prep['chain'],native='(site box)',cofactors=','.join(prep['cofactors']),
                                contacts=np.nan,pocket=prep['pocket'],center=prep['center'],ref_CNN_VS=np.nan,ref_minAff=np.nan,rmsd_top=np.nan,rmsd_min=np.nan,cdist=np.nan,pb_valid=None,pb_failed='',tier='site_only'))
                continue
            prep=prepare_structure(pdb,set(t['ligs']))
            smi=chemcomp_smiles(prep['native'])
            lib=pd.DataFrame([dict(id=prep['native'],smiles=smi)])
            d=dock_set(pdb,lib,f'native_{pdb}')
            if d.empty: raise RuntimeError('no scored poses came back - see the GNINA log lines printed above')
            rmsd_top=rmsd_min=cdist=np.nan
            nat=np.array([_xyz(l) for l in open(WORK/f'{pdb}_native.pdb') if l.startswith('HETATM')])
            poses=[m for m in read_sdf(WORK/'dock'/f'{pdb}_native_{pdb}.sdf') if m.HasProp('CNNscore')]
            top=max(poses,key=lambda m: float(m.GetProp('CNNscore'))*float(m.GetProp('CNNaffinity')))
            heavy=[a.GetIdx() for a in top.GetAtoms() if a.GetAtomicNum()>1]
            cdist=float(np.linalg.norm(top.GetConformer().GetPositions()[heavy].mean(0)-nat.mean(0)))   # centroid shift of the top pose
            try:
                ref=Chem.MolFromPDBFile(str(WORK/f'{pdb}_native.pdb'),sanitize=False,removeHs=True)
                ref=AllChem.AssignBondOrdersFromTemplate(Chem.MolFromSmiles(smi),ref)
                r=[]
                for m in poses:
                    mm=Chem.RemoveHs(m,sanitize=False); mm.UpdatePropertyCache(strict=False); Chem.FastFindRings(mm)
                    r.append((float(m.GetProp('CNNscore'))*float(m.GetProp('CNNaffinity')),rdMolAlign.CalcRMS(mm,ref)))
                rmsd_top=max(r)[1]; rmsd_min=min(x[1] for x in r)
            except Exception as e: print('  rmsd n/a for',pdb,'(',e,') - falling back to centroid shift',round(cdist,2),'A')
            pb_ok,pb_fail=pb_check(pdb,top)
            VAL.append(dict(pdb=pdb,protein=t['protein'],role=t['role'],res=t['res'],chain=prep['chain'],native=prep['native'],cofactors=','.join(prep['cofactors']),
                            contacts=prep['contacts'],pocket=prep['pocket'],center=prep['center'],
                            ref_CNN_VS=d.CNN_VS.iloc[0],ref_minAff=d.minAff.iloc[0],rmsd_top=rmsd_top,rmsd_min=rmsd_min,cdist=cdist,
                            pb_valid=pb_ok,pb_failed='; '.join(pb_fail),native_smiles=smi,tier=''))
        except Exception as e:
            REJ.append(dict(pdb=pdb,protein=t['protein'],reason=f'{type(e).__name__}: {e}'))
            print('REJECTED',pdb,'->',e)
    val=pd.DataFrame(VAL)
    ok,prov=CFG['redock_rmsd_ok'],CFG['redock_rmsd_prov']
    def _tier(r):
        if r.tier=='site_only': return 'site_only'
        if CFG['require_posebusters'] and r.pb_valid is False: return 'rejected'   # physically implausible re-docked pose
        if not np.isnan(r.rmsd_top):  return 'validated' if r.rmsd_top<=ok else ('provisional' if r.rmsd_top<=prov else 'rejected')
        return 'provisional' if r.cdist<=2.0 else 'rejected'     # no RMSD possible -> centroid shift only, never "validated"
    val['tier']=val.apply(_tier,axis=1)
    val['pose_ok']=val.tier=='validated'
    show=['pdb','protein','res','native','contacts','rmsd_top','rmsd_min','cdist','pb_valid','pb_failed','ref_CNN_VS','tier']
    display(val[show].round(2))
    print('\n'.join(f"  not a usable target: {r['pdb']} {r['protein']} - {r['reason']}" for r in REJ))

    V=val[val.tier=='validated']; P=val[val.tier=='provisional']
    USE_T = V if (len(V)>=CFG['min_validated'] and not CFG.get('use_provisional')) else pd.concat([V,P])
    if INCLUDE_SITE_ONLY and (val.tier=='site_only').any():
        S=val[val.tier=='site_only'].copy(); S['ref_CNN_VS']=USE_T.ref_CNN_VS.median(); S['ref_minAff']=USE_T.ref_minAff.median()   # pooled reference: comparable only roughly
        USE_T=pd.concat([USE_T,S]); print('WARNING: active-site-only targets use the pooled median reference score - treat as exploratory.')
    USE_T=USE_T.reset_index(drop=True)
    print(f"\ntargets used: {len(USE_T)}  = {len(V)} validated" + (f" + {len(P)} provisional (too few validated)" if len(V)<CFG['min_validated'] else '') +
          f" | rejected after re-docking: {(val.tier=='rejected').sum()} | failed quality gates: {len(REJ)}")
    if len(USE_T)<3: print('Few validated targets: raise max_resolution a little, add MANUAL_PDBS, or inspect the rejected rows above.')
    USE_T_all = USE_T.copy()

In [ ]:
#@title Step 2 - FORM: prepare structures and validate by re-docking the native ligand { display-mode: "form" }
def build_validate_form():
    with gr.Blocks(title='Step 2 - Validation') as demo:
        gr.Markdown('## Step 2 - Prepare the structures and validate them\nEach protein is prepared (cofactors kept, ligand pocket defined) and its co-crystal ligand is **re-docked**. A target is *validated* only if GNINA reproduces the crystal pose. Weak targets are never used silently.')
        with gr.Row():
            ok = gr.Slider(0.5, 4.0, value=CFG['redock_rmsd_ok'], step=0.1, label='Validated if re-docked RMSD ≤ (Å)')
            prov = gr.Slider(1.0, 6.0, value=CFG['redock_rmsd_prov'], step=0.1, label='Provisional if RMSD ≤ (Å)')
            minv = gr.Slider(1, 10, value=CFG['min_validated'], step=1, label='Use provisional targets only if fewer than N are validated')
        with gr.Row():
            usep = gr.Checkbox(CFG.get('use_provisional', False), label='Always include provisional targets')
            pbq = gr.Checkbox(CFG['require_posebusters'], label='Require a PoseBusters-valid re-docked pose')
            site = gr.Checkbox(INCLUDE_SITE_ONLY, label='Also dock active-site-only proteins (exploratory)')
            redo = gr.Checkbox(False, label='Redo docking (ignore cached results)')
        with gr.Row():
            exh = gr.Slider(4, 32, value=CFG['exhaustiveness'], step=1, label='GNINA exhaustiveness (higher = slower, more thorough)')
            nm = gr.Slider(3, 20, value=CFG['num_modes'], step=1, label='Poses kept per ligand')
            ab = gr.Slider(2, 12, value=CFG['autobox_add'], step=1, label='Box padding around the ligand (Å)')
        with gr.Row():
            run = gr.Button('Prepare structures and re-dock native ligands', variant='primary')
            stop = gr.Button('Stop')
        log = gr.Textbox(label='Log', lines=9, interactive=False)
        msg = gr.Markdown()
        t_val = gr.Dataframe(label='Validation table (pocket = residues lining the ligand)', interactive=False, wrap=True)
        t_rej = gr.Dataframe(label='Not usable (failed the quality checks)', interactive=False, wrap=True)
        pick = gr.CheckboxGroup(label='Targets to dock against (untick to drop any)', choices=[], value=[])
        confirm = gr.Button('Use the ticked targets')
        out = gr.Markdown()

        def _est():
            n = len(targets) if globals().get('targets') is not None else 0
            return f'{n} proteins to prepare. Expected time: about {n * (0.6 if USE_GPU else 3.2):.0f} min ({"GPU" if USE_GPU else "CPU only - switch the Colab runtime to a T4 GPU to be ~5x faster"}). Results are cached, so a rerun is quick.' if n else '**Run step 1 first** (no targets yet).'
        stop.click(lambda: UI.update(stop=True) or 'Stopping after the current protein ...', None, msg)

        def on_run(ok_, prov_, minv_, usep_, pbq_, site_, redo_, exh_, nm_, ab_):
            global INCLUDE_SITE_ONLY
            try: need('targets')
            except Exception as e:
                yield str(e), f'**{e}**', gr.update(), gr.update(), gr.update(); return
            CFG.update(redock_rmsd_ok=float(ok_), redock_rmsd_prov=float(prov_), min_validated=int(minv_), use_provisional=bool(usep_), require_posebusters=bool(pbq_),
                       exhaustiveness=int(exh_), num_modes=int(nm_), autobox_add=int(ab_))
            INCLUDE_SITE_ONLY = bool(site_)
            if redo_:
                for f in (WORK / 'dock').glob('*_native_*.sdf'): f.unlink()
            for cap, state, _ in stream_job(stage_validate):
                if state == 'running': yield cap.text(), 'Working ... ' + _est(), gr.update(), gr.update(), gr.update()
                elif state == 'done':
                    cols = [c for c in ['pdb', 'protein', 'res', 'native', 'contacts', 'rmsd_top', 'cdist', 'pb_valid', 'tier', 'pocket'] if c in val.columns]
                    labels = [(f"{x.pdb} · {str(x.protein)[:40]} · {x.tier}" + (f" (RMSD {x.rmsd_top:.1f} Å)" if x.rmsd_top == x.rmsd_top else ''), x.pdb) for x in USE_T_all.itertuples()]
                    nv = int((val.tier == 'validated').sum()); npv = int((val.tier == 'provisional').sum())
                    yield (cap.text(), f'**{nv} validated, {npv} provisional, {int((val.tier == "rejected").sum())} rejected after re-docking; {len(REJ)} failed the structure checks.** {len(USE_T_all)} target(s) selected for docking.',
                           show_df(val[cols]), show_df(pd.DataFrame(REJ)) if len(REJ) else None, gr.update(choices=labels, value=[v for _, v in labels]))
                else: yield cap.text(), f'**{state}** - see the log', gr.update(), gr.update(), gr.update()
        run.click(on_run, [ok, prov, minv, usep, pbq, site, redo, exh, nm, ab], [log, msg, t_val, t_rej, pick])

        def on_confirm(ids):
            global USE_T
            need('USE_T_all')
            USE_T = USE_T_all[USE_T_all.pdb.isin(ids)].reset_index(drop=True)
            return f'**{len(USE_T)} target(s) will be used for docking.** Next: Step 3 (library).'
        confirm.click(on_confirm, pick, out)
    return demo

run_or_launch('validate', build_validate_form, stage_validate)

## Step 3 - Ligand library

In [ ]:
# --- drug-likeness helpers ---
params=FilterCatalogParams(); params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS); pains=FilterCatalog(params)
pb=FilterCatalogParams(); pb.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK); brenk=FilterCatalog(pb)
try:
    sys.path.append(os.path.join(RDConfig.RDContribDir,'SA_Score')); import sascorer
except Exception: sascorer=None

def props(smi):
    m=Chem.MolFromSmiles(smi)
    return dict(MW=Descriptors.MolWt(m),logP=Crippen.MolLogP(m),TPSA=rdMolDescriptors.CalcTPSA(m),HBD=rdMolDescriptors.CalcNumHBD(m),
                HBA=rdMolDescriptors.CalcNumHBA(m),RotB=rdMolDescriptors.CalcNumRotatableBonds(m),HA=m.GetNumHeavyAtoms(),QED=QED.qed(m),
                SA=(sascorer.calculateScore(m) if sascorer else np.nan),PAINS=int(pains.HasMatch(m)),Brenk=len(brenk.GetMatches(m)))
def annotate(df):
    P=pd.DataFrame([props(s) for s in df.smiles]); return pd.concat([df.reset_index(drop=True),P],axis=1)
def drug_like(df):
    return df[(df.MW<=500)&(df.logP<=5.5)&(df.TPSA<=130)&(df.HBD<=3)&((df.SA<=4.5)|df.SA.isna())]

from collections import defaultdict
# ===== Ligand library: load (many formats / sources) -> standardise -> filter -> pre-screen -> docking set =====
import gzip, hashlib, glob, io
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import DataStructs
from rdkit.SimDivFilters import rdSimDivPickers

PARENTS = {   # demo series (CF3-substituted 2-(benzylidenehydrazinyl)-4-(4-cyanophenyl)thiazoles)
 'Molecule_1':'FC(F)(F)c1ccccc1C=NNc1nc(-c2ccc(C#N)cc2)cs1',
 'Molecule_2':'FC(F)(F)c1cccc(C=NNc2nc(-c3ccc(C#N)cc3)cs2)c1',
 'Molecule_3':'FC(F)(F)c1ccc(C=NNc2nc(-c3ccc(C#N)cc3)cs2)cc1'}

_lfc=rdMolStandardize.LargestFragmentChooser(); _unch=rdMolStandardize.Uncharger()
def standardize(m):
    """strip salts/solvents (largest fragment), neutralise, sanitise"""
    try:
        m=rdMolStandardize.Cleanup(m); m=_lfc.choose(m); m=_unch.uncharge(m); Chem.SanitizeMol(m); return m
    except Exception: return None

_SMI_COLS=('smiles','canonical_smiles','canonicalsmiles','isomeric_smiles','isomericsmiles','smi','connectivitysmiles','structure','molecule')
_ID_COLS=('id','name','compound','compound_id','mol_name','molecule_name','title','molecule_chembl_id','chembl_id','zinc_id','zincid','catalog_id','cid','pref_name')
def _rows_from_table(df,source):
    cols={c.lower().strip():c for c in df.columns}
    sc=next((cols[c] for c in _SMI_COLS if c in cols),None) or next((cols[c] for c in cols if 'smiles' in c),None)
    if sc is None:                                  # guess: column where most of the first 50 values parse as molecules
        best=(0,None)
        for c in df.columns:
            v=df[c].astype(str).head(50); ok=sum(Chem.MolFromSmiles(x) is not None for x in v)
            if ok>best[0]: best=(ok,c)
        sc=best[1]
    if sc is None: raise ValueError(f'{source}: no SMILES column found (columns: {list(df.columns)})')
    ic=next((cols[c] for c in _ID_COLS if c in cols and cols[c]!=sc),None)
    out=[]
    for k,r in enumerate(df.itertuples(index=False)):
        smi=str(getattr(r,'_asdict')()[df.columns[list(df.columns).index(sc)]] if hasattr(r,'_asdict') else r[list(df.columns).index(sc)])
        nm=str(r[list(df.columns).index(ic)]) if ic else f'{Path(source).stem}_{k+1}'
        out.append((nm,smi,None,source))
    return out

def read_any(path):
    """-> list of (id, smiles_or_None, mol_or_None, source). Handles .smi/.smiles/.txt/.csv/.tsv/.xlsx/.sdf/.sdf.gz/.mol/.mol2"""
    p=str(path); low=p.lower(); src=Path(p).name; out=[]
    if low.endswith(('.sdf','.sd','.sdf.gz','.sd.gz')):
        fh=gzip.open(p) if low.endswith('.gz') else open(p,'rb')
        for k,m in enumerate(Chem.ForwardSDMolSupplier(fh,removeHs=False,sanitize=True)):
            if m is None: continue
            nm=m.GetProp('_Name') if m.HasProp('_Name') and m.GetProp('_Name').strip() else next((m.GetProp(x) for x in ('ID','id','Name','NAME','COMPOUND_ID','CHEMBL_ID') if m.HasProp(x)),f'{Path(p).stem}_{k+1}')
            out.append((nm,None,m,src))
    elif low.endswith('.mol2'):
        r=sh(f'obabel "{p}" -osmi',check=False).stdout
        for k,l in enumerate(r.splitlines()):
            t=l.split()
            if t: out.append((t[1] if len(t)>1 else f'{Path(p).stem}_{k+1}',t[0],None,src))
    elif low.endswith(('.mol',)):
        m=Chem.MolFromMolFile(p)
        if m is not None: out.append((Path(p).stem,None,m,src))
    elif low.endswith(('.xlsx','.xls')): out=_rows_from_table(pd.read_excel(p),src)
    elif low.endswith(('.csv','.tsv')): out=_rows_from_table(pd.read_csv(p,sep=None,engine='python'),src)
    else:                                            # .smi / .smiles / .ism / .txt : "SMILES  name"
        lines=[l.strip() for l in open(p,errors='ignore') if l.strip() and not l.startswith('#')]
        if lines and (',' in lines[0] or '\t' in lines[0]) and lines[0].split(',')[0].lower().strip() in _SMI_COLS+_ID_COLS:
            return _rows_from_table(pd.read_csv(p,sep=None,engine='python'),src)
        for k,l in enumerate(lines):
            t=l.replace(',',' ').split()
            if k==0 and t[0].lower() in _SMI_COLS: continue
            out.append((t[1] if len(t)>1 else f'{Path(p).stem}_{k+1}',t[0],None,src))
    return out

# ---------- online sources ----------
def chembl_approved(max_n=2000):
    out=[]; url='https://www.ebi.ac.uk/chembl/api/data/molecule.json'; params=dict(max_phase=4,limit=1000,molecule_structures__canonical_smiles__isnull=False)
    while url and len(out)<max_n:
        r=requests.get(url,params=params,timeout=120); r.raise_for_status(); j=r.json(); params=None
        for m in j['molecules']:
            s=(m.get('molecule_structures') or {}).get('canonical_smiles')
            if s: out.append((m.get('pref_name') or m['molecule_chembl_id'],s,None,'ChEMBL approved'))
        nxt=j['page_meta'].get('next'); url=('https://www.ebi.ac.uk'+nxt) if nxt else None
    return out[:max_n]
def chembl_target_ligands(target_id,pchembl_min=6.0,max_n=2000):
    best={}; url='https://www.ebi.ac.uk/chembl/api/data/activity.json'; params=dict(target_chembl_id=target_id,pchembl_value__gte=pchembl_min,limit=1000)
    while url and len(best)<max_n:
        r=requests.get(url,params=params,timeout=120); r.raise_for_status(); j=r.json(); params=None
        for a in j['activities']:
            s=a.get('canonical_smiles');
            if s: best[a['molecule_chembl_id']]=(a['molecule_chembl_id'],s,None,f'ChEMBL {target_id}')
        nxt=j['page_meta'].get('next'); url=('https://www.ebi.ac.uk'+nxt) if nxt else None
    return list(best.values())[:max_n]
def _pc_props(j):
    return [(str(p.get('Title') or f"CID{p['CID']}"),next((v for k,v in p.items() if 'SMILES' in k),None),None,'PubChem') for p in j['PropertyTable']['Properties']]
def pubchem_similarity(smiles,threshold=85,max_n=500):
    url='https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/similarity/smiles/property/SMILES,Title/JSON'
    r=requests.post(url,data={'smiles':smiles},params=dict(Threshold=threshold,MaxRecords=max_n),timeout=180); j=r.json()
    for _ in range(30):                                   # asynchronous job -> poll
        if 'Waiting' in j:
            time.sleep(4); r=requests.get(f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/listkey/{j['Waiting']['ListKey']}/property/SMILES,Title/JSON",timeout=180); j=r.json()
        else: break
    if 'PropertyTable' not in j: raise RuntimeError(f'PubChem returned: {str(j)[:200]}')
    return _pc_props(j)
def pubchem_names(names):
    out=[]
    for n in names:
        try:
            r=requests.get(f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{requests.utils.quote(n)}/property/SMILES,Title/JSON',timeout=60)
            if r.ok: x=_pc_props(r.json())[0]; out.append((n,x[1],None,'PubChem'))
            else: print('  PubChem: not found',n)
        except Exception as e: print('  PubChem error',n,e)
    return out
def download_url(url):
    f=WORK/'library_in'/Path(url.split('?')[0]).name; f.parent.mkdir(exist_ok=True)
    r=requests.get(url,timeout=300); r.raise_for_status(); f.write_bytes(r.content); return read_any(f)

def upload_files():
    if UPLOADED_FILES: return list(UPLOADED_FILES)
    try:
        from google.colab import files
        up=files.upload(); d=WORK/'library_in'; d.mkdir(exist_ok=True); paths=[]
        for n,b in up.items(): (d/n).write_bytes(b); paths.append(str(d/n))
        return paths
    except ImportError:
        print('not running in Colab - put files in',WORK/'library_in','or set LIB_PATH'); return glob.glob(str(WORK/'library_in'/'*'))

def load_raw_library():
    src=LIBRARY_SOURCE; rows=[]
    if src.startswith('Upload'):
        paths=upload_files()+[p for pat in LIB_PATH.split(',') if pat.strip() for p in glob.glob(pat.strip())]
        for p in paths:
            try: rows+=read_any(p); print('  read',Path(p).name,'->',sum(1 for r in rows if r[3]==Path(p).name),'records')
            except Exception as e: print('  could not read',p,type(e).__name__,e)
    elif src.startswith('Paste'): rows=[(t[1] if len(t)>1 else f'pasted_{k+1}',t[0],None,'pasted') for k,t in enumerate(l.replace(',',' ').split() for l in LIB_PASTE.splitlines() if l.strip() and not l.startswith('#'))]
    elif src.startswith('ChEMBL approved'): rows=chembl_approved(LIB_MAX_MOLECULES*4)
    elif src.startswith('ChEMBL ligands'): rows=chembl_target_ligands(CHEMBL_TARGET_ID,max_n=LIB_MAX_MOLECULES*4)
    elif src.startswith('PubChem similarity'): rows=pubchem_similarity(QUERY_SMILES,max_n=LIB_MAX_MOLECULES*2)
    elif src.startswith('PubChem names'): rows=pubchem_names([x.strip() for x in PUBCHEM_NAMES.split(',') if x.strip()])
    elif src.startswith('Download'): rows=download_url(LIB_URL)
    else: rows=[(k,v,None,'demo') for k,v in PARENTS.items()]
    return rows

def prepare_library(rows,min_mw,max_mw,max_heavy=70):
    seen={}; keep=[]; drop=defaultdict(int)
    for nm,smi,mol,src in rows:
        m=mol if mol is not None else (Chem.MolFromSmiles(smi) if smi else None)
        if m is None: drop['unparseable']+=1; continue
        m=standardize(Chem.RemoveHs(m))
        if m is None: drop['standardisation failed']+=1; continue
        mw=Descriptors.MolWt(m)
        if m.GetNumHeavyAtoms()>max_heavy or not (min_mw<=mw<=max_mw): drop[f'MW outside {min_mw:.0f}-{max_mw:.0f} / >{max_heavy} heavy atoms']+=1; continue
        ik=Chem.MolToInchiKey(m)[:14]
        if ik in seen: drop['duplicate']+=1; continue
        seen[ik]=1; keep.append((str(nm),Chem.MolToSmiles(m),src))
    df=pd.DataFrame(keep,columns=['name','smiles','source'])
    df.insert(0,'id',[re.sub(r'[^A-Za-z0-9_.-]+','_',n)[:40] or f'mol{i}' for i,n in enumerate(df.name)])
    dup=df.id.duplicated(keep=False); df.loc[dup,'id']=[f'{i}_{k}' for k,i in enumerate(df.id[dup])]   # unique ids
    print(f'{len(rows)} records read -> {len(df)} usable molecules'+('' if not drop else ' | dropped: '+', '.join(f'{v} {k}' for k,v in drop.items())))
    return df

def prescreen(df,n,mode,native_smiles=()):
    if len(df)<=n: return df
    if mode.startswith('similarity') and native_smiles:
        ref=[AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s),2,2048) for s in native_smiles if Chem.MolFromSmiles(s)]
        sc=[max(DataStructs.BulkTanimotoSimilarity(AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s),2,2048),ref)) for s in df.smiles]
        out=df.assign(sim_native=sc).sort_values('sim_native',ascending=False).head(n); print(f'pre-screen: kept the {n} molecules most similar to the co-crystal ligands (Tanimoto {out.sim_native.min():.2f}-{out.sim_native.max():.2f})'); return out
    if mode.startswith('diverse') and len(df)>n:
        fps=[AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s),2,2048) for s in df.smiles]
        idx=list(rdSimDivPickers.MaxMinPicker().LazyBitVectorPick(fps,len(fps),n,seed=CFG['seed'])); print(f'pre-screen: MaxMin diverse subset of {n}'); return df.iloc[idx]
    print(f'library truncated to the first {n} molecules (set LIB_PRESCREEN or raise LIB_MAX_MOLECULES)'); return df.head(n)

def library_tag(df): return 'lib_'+hashlib.md5(('|'.join(df.id+df.smiles)).encode()).hexdigest()[:8]     # new library -> new docking cache

In [ ]:
def stage_library():
    global LIB, LIB_ALL, LIB_TAG, RUN_SAR, _canon, has_parents, mode, n_t, native_smiles, rows
    rows=load_raw_library()
    LIB_ALL=prepare_library(rows,MIN_MW,MAX_MW)
    native_smiles=[s for s in (USE_T if USE_T is not None else pd.DataFrame()).get('native_smiles',pd.Series(dtype=object)) if isinstance(s,str) and s]
    mode=LIB_PRESCREEN.split(' ')[0] if not LIB_PRESCREEN.startswith('none') else 'none'
    LIB=prescreen(LIB_ALL,LIB_MAX_MOLECULES,mode,native_smiles).reset_index(drop=True)
    LIB_TAG=library_tag(LIB)
    n_t=int((USE_T.role=='target').sum()) if USE_T is not None else 0
    print(f'\nlibrary to dock: {len(LIB)} molecules x {n_t} validated targets = {len(LIB)*n_t} dockings (~{len(LIB)*n_t*(3 if USE_GPU else 25)/3600:.1f} h estimated; reduce LIB_MAX_MOLECULES if too long)')
    _canon=lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s))
    has_parents=any(_canon(s) in set(LIB_ALL.smiles) for s in PARENTS.values())
    RUN_SAR = SAR_MODE.startswith('on') or (SAR_MODE.startswith('auto') and has_parents)
    print('SAR module:', 'ON (thiazole-hydrazone enumeration will run)' if RUN_SAR else 'OFF (library is not the demo series - sections 8-11 are skipped)')
    display(LIB.head(10))

In [ ]:
#@title Step 3 - FORM: ligand library (upload / paste / ChEMBL / PubChem / URL) { display-mode: "form" }
def build_library_form():
    with gr.Blocks(title='Step 3 - Library') as demo:
        gr.Markdown('## Step 3 - Ligand library\nLoad molecules from files or online sources. They are standardised (salts removed, neutralised), de-duplicated and filtered; large libraries are pre-screened down to what you can afford to dock.')
        src = gr.Dropdown(LIB_SOURCES, value=LIBRARY_SOURCE if LIBRARY_SOURCE in LIB_SOURCES else LIB_SOURCES[0], label='Source')
        files = gr.File(label='Upload molecule files: .smi .txt .csv .tsv .xlsx .sdf .sdf.gz .mol2 .mol (several allowed)', file_count='multiple')
        paste = gr.Textbox(value=LIB_PASTE.strip(), lines=5, label='Or paste SMILES (one per line, optional name after a space)')
        with gr.Row():
            cid = gr.Textbox(value=CHEMBL_TARGET_ID, label='ChEMBL target id (e.g. CHEMBL1900) for "ChEMBL ligands of a target"')
            qs = gr.Textbox(value=QUERY_SMILES, label='Query SMILES for "PubChem similarity search"')
        with gr.Row():
            names = gr.Textbox(value=PUBCHEM_NAMES, label='Compound names (comma-separated) for "PubChem compound names"')
            url = gr.Textbox(value=LIB_URL, label='URL of a molecule file for "Download from URL"')
        with gr.Row():
            mw1 = gr.Slider(50, 500, value=MIN_MW, step=10, label='Min molecular weight')
            mw2 = gr.Slider(200, 1000, value=MAX_MW, step=10, label='Max molecular weight')
            nmax = gr.Slider(10, 5000, value=LIB_MAX_MOLECULES, step=10, label='Max molecules to dock')
            pre = gr.Dropdown(["similarity to native ligands", "diverse subset", "none (first N)"], value=LIB_PRESCREEN, label='If the library is larger')
        run = gr.Button('Load and clean the library', variant='primary')
        msg = gr.Markdown()
        log = gr.Textbox(label='Log', lines=7, interactive=False)
        tbl = gr.Dataframe(label='Library to dock (first 300 rows)', interactive=False, wrap=True)
        grid = gr.Image(label='First molecules', type='pil')
        dl = gr.File(label='Cleaned library (CSV)', interactive=False)

        def on_run(s, fl, p, ci, q, nms, u, a, b, n, pr):
            paths = []
            for f in (fl or []):
                f = getattr(f, 'name', f); dst = WORK / 'library_in'; dst.mkdir(exist_ok=True)
                tgt = dst / Path(f).name; shutil.copy(f, tgt); paths.append(str(tgt))
            UPLOADED_FILES[:] = paths
            if s.startswith('Upload') and not paths:
                yield '', '**Choose at least one file to upload** (or pick another source).', gr.update(), gr.update(), gr.update(); return
            set_library_settings(s, p, ci, q, nms, u, a, b, n, pr)
            for cap, state, _ in stream_job(stage_library):
                if state == 'running': yield cap.text(), 'Loading ...', gr.update(), gr.update(), gr.update()
                elif state == 'done':
                    f = WORK / 'library_clean.csv'; LIB.to_csv(f, index=False)
                    yield (cap.text(), f'**{len(LIB)} molecules ready** (from {len(LIB_ALL)} usable). ' + ('The thiazole SAR demo module is available in "run everything" mode.' if RUN_SAR else ''),
                           show_df(LIB.head(300)), mol_grid(LIB.smiles, LIB.id), str(f))
                else: yield cap.text(), f'**{state}** - see the log', gr.update(), gr.update(), gr.update()
        run.click(on_run, [src, files, paste, cid, qs, names, url, mw1, mw2, nmax, pre], [log, msg, tbl, grid, dl])
    return demo

run_or_launch('library', build_library_form, stage_library)

## Step 4 - Dock

In [ ]:
def dock_all(lib,tag,tgts):
    res=[]
    for t in tqdm(tgts.to_dict('records'),desc=tag):
        d=dock_set(t['pdb'],lib,tag)
        if len(d):
            d['ref']=t['ref_CNN_VS']; d['rel']=d.CNN_VS/d.ref; d['protein']=t['protein']; d['role']=t['role']; res.append(d)
    return pd.concat(res,ignore_index=True) if res else pd.DataFrame()

def stage_dock():
    global lib_agg, plt, pv, res_lib, smiles_map, sns, top
    res_lib=dock_all(LIB[['id','smiles']],LIB_TAG,USE_T[USE_T.role=='target'])
    assert len(res_lib),'no docking results - see the GNINA messages above'
    import seaborn as sns, matplotlib.pyplot as plt
    lib_agg=res_lib.groupby('id').agg(mean_rel=('rel','mean'),max_rel=('rel','max'),n_targets=('pdb','nunique'),CNNaff=('CNNaff','mean'),minAff=('minAff','mean')).sort_values('mean_rel',ascending=False)
    top=lib_agg.head(30).index
    pv=res_lib[res_lib.id.isin(top)].pivot_table(index='id',columns='pdb',values='rel').reindex(top)
    plt.figure(figsize=(max(5,0.7*pv.shape[1]+2),max(3,0.3*len(pv)+1))); sns.heatmap(pv,annot=len(pv)<=15,fmt='.2f',cmap='viridis'); plt.title('CNN_VS relative to the native ligand (top 30 molecules)'); plt.show()
    display(lib_agg.head(30).round(2))
    print('docked molecules per target:'); display(res_lib.groupby(['pdb','protein']).id.nunique().to_frame('n_molecules'))
    smiles_map = dict(zip(LIB.id, LIB.smiles))

In [ ]:
#@title Step 4 - FORM: dock the library on every selected target { display-mode: "form" }
def build_dock_form():
    with gr.Blocks(title='Step 4 - Docking') as demo:
        gr.Markdown('## Step 4 - Dock the library on the selected targets\nEvery molecule is docked on every target; its score is divided by the score of that protein\'s own crystal ligand (`rel` = 1.0 means "as good as the crystal ligand").')
        with gr.Row():
            exh = gr.Slider(4, 32, value=CFG['exhaustiveness'], step=1, label='Exhaustiveness')
            nm = gr.Slider(3, 20, value=CFG['num_modes'], step=1, label='Poses per ligand')
            maxn = gr.Slider(10, 5000, value=LIB_MAX_MOLECULES, step=10, label='Dock at most this many molecules (the first N of the library)')
        with gr.Row():
            run = gr.Button('Start docking', variant='primary'); stop = gr.Button('Stop')
        log = gr.Textbox(label='Progress', lines=8, interactive=False)
        msg = gr.Markdown()
        heat = gr.Image(label='CNN_VS relative to the native ligand (top molecules x targets)', type='pil')
        agg = gr.Dataframe(label='Best molecules (mean over targets)', interactive=False, wrap=True)
        per = gr.Dataframe(label='Molecules docked per target', interactive=False)

        def _est():
            try: need('USE_T', 'LIB')
            except Exception as e: return f'**{e}**'
            n_t = int((USE_T.role == 'target').sum())
            return f'{len(LIB)} molecules x {n_t} targets = **{len(LIB) * n_t} dockings**, about {len(LIB) * n_t * (3 if USE_GPU else 25) / 3600:.1f} h on {"the GPU" if USE_GPU else "CPU only (a T4 GPU runtime is much faster)"}. Finished dockings are cached.'
        stop.click(lambda: UI.update(stop=True) or 'Stopping after the current target ...', None, msg)

        def on_run(e_, n_, m_):
            global LIB
            try: need('USE_T', 'LIB')
            except Exception as e: yield str(e), f'**{e}**', gr.update(), gr.update(), gr.update(); return
            CFG.update(exhaustiveness=int(e_), num_modes=int(n_)); LIB = LIB.head(int(m_)).reset_index(drop=True)
            for cap, state, _ in stream_job(stage_dock):
                if state == 'running': yield cap.text(), 'Docking ... ' + _est(), gr.update(), gr.update(), gr.update()
                elif state == 'done':
                    yield (cap.text(), f'**Done:** {len(res_lib)} scored poses for {res_lib.id.nunique()} molecules on {res_lib.pdb.nunique()} targets. Next: Step 5 (shortlist).',
                           cap.images[0] if cap.images else None, show_df(lib_agg.head(30).reset_index()),
                           show_df(res_lib.groupby(['pdb', 'protein']).id.nunique().to_frame('n_molecules').reset_index()))
                else: yield cap.text(), f'**{state}** - see the progress box', gr.update(), gr.update(), gr.update()
        run.click(on_run, [exh, nm, maxn], [log, msg, heat, agg, per])
    return demo

run_or_launch('dock', build_dock_form, stage_dock)

*(Built-in demo: CF3-thiazole hydrazone series - generation-1/2 enumeration and Free-Wilson on docking scores. Runs only in 'run everything' mode when the library contains the demo parents. The general SAR tool is step 7.)*

In [ ]:
def stage_demo_sar():
    global ARX, BASE, C5, D, HET, LINK, LT, M, Ridge, SITES, SUBS, TT, Xf, _atom, agg1, agg2, base_id, base_on_TT, base_rel, build_smiles, canon, chk, coef, combos, d, design_matrix, fw, gen1, gen1_designs, gen2, gen2_all, k, known, make_df, pdb, r, res1, res2, rows, rr, s, space, sub, top_opts, v
    if RUN_SAR:
        SUBS = {'H':'','F':'F','Cl':'Cl','Br':'Br','CF3':'C(F)(F)F','OCF3':'OC(F)(F)F','OMe':'OC','OH':'O','Me':'C','NO2':'[N+](=O)[O-]',
                'CN':'C#N','NMe2':'N(C)C','CONH2':'C(N)=O','SO2Me':'S(C)(=O)=O','tBu':'C(C)(C)C','aza':'N'}
        ARX  = {k:v for k,v in SUBS.items() if k not in ('aza','NMe2','CONH2','SO2Me')}
        HET  = {'S':'s','O':'o','NH':'[nH]'}
        C5   = {'H':'','Me':'C','Cl':'Cl'}
        LINK = {'H':'C=NN','Me':'C(C)=NN'}
        BASE = dict(o1='H',m1='H',p='H',m2='H',X='CN',het='S',c5='H',link='H')

        def _atom(k):
            s=SUBS[k]
            if k=='aza': return 'n'
            return 'c'+(f'({s})' if s else '')
        def build_smiles(d):
            x=ARX[d['X']]; xb=f'({x})' if x else ''
            c5=C5[d['c5']]; c5b=f'({c5})' if c5 else ''
            tail=f"{LINK[d['link']]}c1nc(-c2ccc{xb}cc2)c{c5b}{HET[d['het']]}1"
            ring=f"c4({tail})"+_atom(d['o1'])+_atom(d['m1'])+_atom(d['p'])+_atom(d['m2'])+'c4'
            return ring
        def canon(smi):
            m=Chem.MolFromSmiles(smi); return Chem.MolToSmiles(m) if m else None

        # sanity: reproduces the three original compounds
        for k,d in dict(Molecule_1=dict(BASE,o1='CF3'),Molecule_2=dict(BASE,m1='CF3'),Molecule_3=dict(BASE,p='CF3')).items():
            assert canon(build_smiles(d))==canon(PARENTS[k]),k
        print('builder reproduces parents OK')

        def make_df(designs,prefix):
            seen={}; rows=[]
            for d in designs:
                s=canon(build_smiles(d))
                if s is None or s in seen: continue
                seen[s]=1; rows.append(dict(d,smiles=s))
            df=pd.DataFrame(rows); df.insert(0,'id',[f'{prefix}{i:04d}' for i in range(len(df))]); return df

        def gen1_designs():
            D=[dict(BASE)]
            for s in ('o1','m1','p'):    D+=[dict(BASE,**{s:v}) for v in SUBS if v!='H']
            D+=[dict(BASE,X=v) for v in ARX if v!='CN']
            D+=[dict(BASE,het=v) for v in HET if v!='S']
            D+=[dict(BASE,c5=v) for v in C5 if v!='H']
            D+=[dict(BASE,link=v) for v in LINK if v!='H']
            return D
        gen1=make_df(gen1_designs(),'G1_')
        print(len(gen1),'generation-1 analogs')
    if RUN_SAR:
        # --- gated SAR cell (property annotation of generation-1) ---
        gen1=annotate(gen1)
        display(gen1[['id','o1','m1','p','X','het','c5','link','MW','logP','TPSA','QED','SA','PAINS']].head(8).round(2))
        print('drug-like:',len(drug_like(gen1)),'/',len(gen1))
    if RUN_SAR:
        LT=USE_T[USE_T.role=='target']
        res1=dock_all(gen1[['id','smiles']],'gen1',LT)
        res1=res1.merge(gen1.drop(columns=['smiles']),on='id')
        res1['LE']=res1.CNNaff/res1.HA
        agg1=res1.groupby('id').agg(mean_rel=('rel','mean'),max_rel=('rel','max'),n_ge1=('rel',lambda x:(x>=1).sum()),CNNaff=('CNNaff','mean'),minAff=('minAff','mean')).reset_index().merge(gen1,on='id')
        display(agg1.sort_values('mean_rel',ascending=False)[['id','o1','m1','p','X','het','c5','link','mean_rel','max_rel','n_ge1','CNNaff','logP','QED']].head(15).round(2))
    if RUN_SAR:
        SITES=['o1','m1','p','X','het','c5','link']
        base_id=gen1[(gen1[list(BASE)]==pd.Series(BASE)).all(axis=1)].id.iloc[0]
        base_rel=res1[res1.id==base_id].set_index('pdb').rel
        rows=[]
        for s in SITES:
            sub=gen1[(gen1[[c for c in BASE if c!=s]]==pd.Series({c:v for c,v in BASE.items() if c!=s})).all(axis=1)]
            for r in sub.itertuples():
                rr=res1[res1.id==r.id].set_index('pdb').rel
                for pdb,v in (rr-base_rel).dropna().items(): rows.append(dict(site=s,option=getattr(r,s),pdb=pdb,d_rel=v))
        D=pd.DataFrame(rows); D['key']=D.site+':'+D.option
        M=D.pivot_table(index='key',columns='pdb',values='d_rel')
        M['mean']=M.mean(axis=1); M=M.sort_values('mean',ascending=False)
        plt.figure(figsize=(max(6,0.5*M.shape[1]),max(6,0.28*len(M)))); sns.heatmap(M.drop(columns='mean'),cmap='RdBu_r',center=0,yticklabels=True); plt.title('Δ relative CNN_VS vs unsubstituted parent'); plt.show()

        # Free-Wilson ridge
        from sklearn.linear_model import Ridge
        def design_matrix(df):
            X=pd.DataFrame(index=df.index)
            for s in SITES:
                for v in sorted(set(gen1[s])):
                    if v!=BASE[s]: X[f'{s}:{v}']=(df[s]==v).astype(float)
            return X
        Xf=design_matrix(agg1); fw=Ridge(alpha=0.3).fit(Xf,agg1.mean_rel)
        coef=pd.Series(fw.coef_,index=Xf.columns).sort_values(ascending=False)
        print('Free-Wilson R2 (train):',round(fw.score(Xf,agg1.mean_rel),2))
        display(coef.round(3).to_frame('contribution to mean_rel').head(12)); display(coef.round(3).tail(6).to_frame('contribution'))

        # lipophilicity / efficiency check
        plt.figure(figsize=(5,4)); plt.scatter(agg1.logP,agg1.mean_rel,c=agg1.QED,cmap='viridis'); plt.colorbar(label='QED'); plt.xlabel('cLogP'); plt.ylabel('mean rel CNN_VS'); plt.show()
        print('corr(logP, score) =',round(agg1[['logP','mean_rel']].corr().iloc[0,1],2),'-> if high, favour ligand-efficient / lower-logP analogs')
    if RUN_SAR:
        def top_opts(site,k,allow_base=True):
            c=coef[[i for i in coef.index if i.startswith(site+':')]]
            opts=[i.split(':',1)[1] for i in c[c>0].sort_values(ascending=False).index[:k]]
            return ([BASE[site]] if allow_base else [])+opts
        space=dict(o1=top_opts('o1',3),m1=top_opts('m1',3),p=top_opts('p',3),m2=[BASE['m2']]+top_opts('m1',1),
                   X=top_opts('X',3)+([] if 'CN' in top_opts('X',3) else ['CN']),het=top_opts('het',2),c5=top_opts('c5',1),link=top_opts('link',1))
        print({k:v for k,v in space.items()})
        combos=[dict(zip(space,v)) for v in itertools.product(*space.values())]
        gen2_all=make_df(combos,'G2_'); gen2_all=annotate(gen2_all)
        gen2_all['FW_pred']=fw.predict(design_matrix(gen2_all).reindex(columns=Xf.columns,fill_value=0))
        gen2_all=drug_like(gen2_all)
        known=set(gen1.smiles)
        gen2_all=gen2_all[~gen2_all.smiles.isin(known)].sort_values('FW_pred',ascending=False)
        # crude diversity: at most 6 per (X,het,link) bucket
        gen2=gen2_all.groupby(['X','het','link'],group_keys=False).head(max(6,CFG['n_gen2']//12)).head(CFG['n_gen2']).reset_index(drop=True)
        print(len(gen2_all),'drug-like combos ->',len(gen2),'selected for docking')

        TT=LT.assign(m=lambda d:d.pdb.map(res1.groupby('pdb').rel.mean())).sort_values('m',ascending=False).head(CFG['n_top_targets'])
        print('gen-2 targets:',list(TT.pdb))
        res2=dock_all(gen2[['id','smiles']],'gen2',TT).merge(gen2.drop(columns=['smiles']),on='id'); res2['LE']=res2.CNNaff/res2.HA
        agg2=res2.groupby('id').agg(mean_rel=('rel','mean'),max_rel=('rel','max'),n_ge1=('rel',lambda x:(x>=1).sum()),CNNaff=('CNNaff','mean')).reset_index().merge(gen2,on='id')
    if RUN_SAR:
        chk=agg2.copy()
        # compare with gen-1 baseline on the same targets
        base_on_TT=res1[res1.pdb.isin(TT.pdb)].groupby('id').rel.mean()
        chk['gen1_best_same_targets']=base_on_TT.max()
        chk['FW_error']=chk.mean_rel-(chk.FW_pred)
        print('FW-predicted vs docked  r =',round(chk[['FW_pred','mean_rel']].corr().iloc[0,1],2),'| median |error| =',round(chk.FW_error.abs().median(),2))
        plt.figure(figsize=(4.5,4)); plt.scatter(chk.FW_pred,chk.mean_rel,s=12); plt.xlabel('Free-Wilson prediction'); plt.ylabel('docked mean rel (gen-2)'); plt.show()
        print('gen-1 best mean rel (same targets): %.2f | gen-2 best: %.2f'%(base_on_TT.max(),agg2.mean_rel.max()))

if AUTO and RUN_SAR: stage_demo_sar()

## Step 5 - Shortlist

In [ ]:
def stage_shortlist():
    global R_ALL, Rall, _t1, aggall, allagg, b, cols, fail, final, g, mols, nm, ok, pbrows, r, sar_cols, sar_info, smiles_map, src_map, sub
    smiles_map=dict(zip(LIB.id,LIB.smiles)); src_map={i:'library' for i in LIB.id}
    sar_cols=['o1','m1','p','m2','X','het','c5','link']; sar_info=[]
    for nm in ('gen1','gen2'):
        if nm in globals():
            g=globals()[nm]; smiles_map.update(dict(zip(g.id,g.smiles))); src_map.update({i:nm for i in g.id}); sar_info.append(g[['id']+sar_cols])
    R_ALL=pd.concat([r for r in (res_lib,globals().get('res1'),globals().get('res2')) if r is not None and len(r)],ignore_index=True)
    aggall=R_ALL.groupby('id').agg(mean_rel=('rel','mean'),max_rel=('rel','max'),n_ge1=('rel',lambda x:(x>=1).sum()),n_targets=('pdb','nunique'),CNNaff=('CNNaff','mean'),minAff=('minAff','mean')).reset_index()
    _t1=R_ALL.sort_values('rel',ascending=False).drop_duplicates('id').set_index('id')
    aggall['best_target']=aggall.id.map(_t1.pdb); aggall['best_protein']=aggall.id.map(_t1.protein)
    aggall['smiles']=aggall.id.map(smiles_map); aggall['source']=aggall.id.map(src_map)
    if sar_info: aggall=aggall.merge(pd.concat(sar_info).drop_duplicates('id'),on='id',how='left')
    aggall=annotate(aggall)
    # priority score: docking vs native, breadth over targets, penalties for lipophilicity and PAINS
    aggall['priority']=aggall.mean_rel+0.5*aggall.n_ge1/max(1,aggall.n_ge1.max())-0.1*np.clip(aggall.logP-4,0,None)-0.15*aggall.PAINS
    final=(drug_like(aggall) if SHORTLIST_DRUGLIKE else aggall).sort_values('priority',ascending=False).head(N_SHORTLIST).reset_index(drop=True)
    cols=['id','source','smiles','best_target','best_protein','mean_rel','max_rel','n_ge1','n_targets','CNNaff','minAff','MW','logP','TPSA','QED','SA','PAINS','Brenk','priority']+([c for c in sar_cols if c in aggall.columns] if sar_info else [])
    allagg=aggall
    display(final[cols].round(2))
    mols=[Chem.MolFromSmiles(s) for s in final.smiles]
    display(Draw.MolsToGridImage(mols[:12],molsPerRow=4,subImgSize=(260,220),legends=[f'{r.id} rel={r.mean_rel:.2f}' for r in final.head(12).itertuples()]))
    print('per-target summary:'); display(R_ALL.groupby(['pdb','protein']).agg(n_molecules=('id','nunique'),mean_rel=('rel','mean'),best_rel=('rel','max'),n_rel_ge1=('rel',lambda x:(x>=1).sum())).round(2).sort_values('best_rel',ascending=False))

    Rall=R_ALL
    pbrows=[]
    for r in tqdm(final.itertuples(),total=len(final)):
        sub=Rall[Rall.id==r.id]
        if sub.empty or not SHORTLIST_PB: pbrows.append((None,'no pose' if sub.empty else 'not run')); continue
        b=sub.sort_values('rel',ascending=False).iloc[0]
        ok,fail=pb_check(b.pdb,Chem.MolFromMolBlock(b.molblock,removeHs=False,sanitize=False))
        pbrows.append((ok,'; '.join(fail)))
    final['pb_valid']=[x[0] for x in pbrows]; final['pb_failed']=[x[1] for x in pbrows]
    display(final[['id','smiles','mean_rel','pb_valid','pb_failed']].round(2))
    print('PoseBusters-valid shortlist:',int((final.pb_valid==True).sum()),'of',len(final))

In [ ]:
#@title Step 5 - FORM: shortlist (potency proxy + breadth + drug-likeness + PoseBusters) { display-mode: "form" }
def build_shortlist_form():
    with gr.Blocks(title='Step 5 - Shortlist') as demo:
        gr.Markdown('## Step 5 - Shortlist\nRanks every docked molecule by `priority` = mean relative score + breadth over targets - penalties (high logP, PAINS), keeps only drug-like ones and re-checks the top poses with PoseBusters.')
        with gr.Row():
            n = gr.Slider(5, 100, value=N_SHORTLIST, step=1, label='Shortlist size')
            dl = gr.Checkbox(True, label='Keep only drug-like molecules (Lipinski/Veber-type filter of the notebook)')
            pb = gr.Checkbox(True, label='Check the best pose of each with PoseBusters')
        run = gr.Button('Build the shortlist', variant='primary')
        log = gr.Textbox(label='Log', lines=5, interactive=False)
        msg = gr.Markdown()
        tbl = gr.Dataframe(label='Shortlist', interactive=False, wrap=True)
        grid = gr.Image(label='Structures', type='pil')
        per = gr.Dataframe(label='Per-target summary', interactive=False)

        def on_run(n_, dl_, pb_):
            global N_SHORTLIST, SHORTLIST_DRUGLIKE, SHORTLIST_PB
            try: need('res_lib')
            except Exception as e: yield str(e), f'**{e}**', gr.update(), gr.update(), gr.update(); return
            N_SHORTLIST, SHORTLIST_DRUGLIKE, SHORTLIST_PB = int(n_), bool(dl_), bool(pb_)
            for cap, state, _ in stream_job(stage_shortlist):
                if state == 'running': yield cap.text(), 'Working ...', gr.update(), gr.update(), gr.update()
                elif state == 'done':
                    c = [x for x in ['id', 'source', 'smiles', 'best_target', 'best_protein', 'mean_rel', 'max_rel', 'n_ge1', 'n_targets', 'CNNaff', 'MW', 'logP', 'TPSA', 'QED', 'SA', 'PAINS', 'Brenk', 'priority', 'pb_valid', 'pb_failed'] if x in final.columns]
                    pt = R_ALL.groupby(['pdb', 'protein']).agg(n_molecules=('id', 'nunique'), mean_rel=('rel', 'mean'), best_rel=('rel', 'max'), n_rel_ge1=('rel', lambda x: (x >= 1).sum())).round(2).reset_index()
                    yield (cap.text(), f'**{len(final)} molecules shortlisted.** Next: Step 6 (pose evaluation).', show_df(final[c]), mol_grid(final.smiles, [f'{r.id} {r.mean_rel:.2f}' for r in final.itertuples()], n=24), show_df(pt))
                else: yield cap.text(), f'**{state}** - see the log', gr.update(), gr.update(), gr.update()
        run.click(on_run, [n, dl, pb], [log, msg, tbl, grid, per])
    return demo

run_or_launch('shortlist', build_shortlist_form, stage_shortlist)

## Step 6 - Pose evaluation

In [ ]:
from scipy.spatial import cKDTree
def contacts(pdb,molblock,cut=4.0):
    lig=Chem.MolFromMolBlock(molblock,removeHs=False); xyz=lig.GetConformer().GetPositions(); el=[a.GetSymbol() for a in lig.GetAtoms()]
    R=[(l[17:20].strip(),l[22:26].strip(),l[12:16].strip(),np.array([float(l[30:38]),float(l[38:46]),float(l[46:54])])) for l in open(WORK/f'{pdb}_receptor.pdb') if l.startswith(('ATOM','HETATM'))]
    P=np.array([r[3] for r in R]); tree=cKDTree(P); out={}
    for i,(x,e) in enumerate(zip(xyz,el)):
        if e=='H': continue
        for j in tree.query_ball_point(x,cut):
            d=float(np.linalg.norm(P[j]-x)); k=(R[j][0],R[j][1])
            polar=(e in 'NOF') and R[j][2][0] in 'NO' and d<=3.5
            if k not in out or d<out[k]['d']: out[k]=dict(res=f'{R[j][0]}{R[j][1]}',d=round(d,2),lig_atom=e,rec_atom=R[j][2],polar=polar)
    return pd.DataFrame(out.values()).sort_values('d')

def show_pose(pdb,molblock,cut=4.0):
    import py3Dmol
    v=py3Dmol.view(width=800,height=550)
    v.addModel((WORK/f'{pdb}_receptor.pdb').read_text(),'pdb'); v.setStyle({'model':0},{'cartoon':{'color':'lightgray','opacity':.8}})
    v.addModel(molblock,'sdf'); v.setStyle({'model':1},{'stick':{'colorscheme':'greenCarbon','radius':.2}})
    sel={'and':[{'within':{'distance':cut,'model':1}},{'model':0}]}
    v.addStyle(sel,{'stick':{'colorscheme':'cyanCarbon','radius':.12}}); v.addResLabels(sel,{'fontSize':9,'backgroundOpacity':.5})
    v.zoomTo({'model':1}); return v

In [ ]:
# ===== Full pose evaluation: interactions, 2D map, 3D view, pose consensus, native-site comparison =====
import io, math, glob
from collections import defaultdict
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem import rdDepictor

ICOL = {'H-bond':'#1f77b4','Salt bridge':'#d62728','Hydrophobic':'#2ca02c','Pi stacking':'#ff7f0e',
        'Cation-pi':'#9467bd','Halogen/F contact':'#e377c2','Metal/heme':'#000000'}
# ---- protein atom typing (heavy atoms) ----
_DON = {('*','N'),('SER','OG'),('THR','OG1'),('TYR','OH'),('ASN','ND2'),('GLN','NE2'),('LYS','NZ'),('ARG','NE'),('ARG','NH1'),('ARG','NH2'),('HIS','ND1'),('HIS','NE2'),('TRP','NE1'),('CYS','SG')}
_ACC = {('*','O'),('*','OXT'),('ASP','OD1'),('ASP','OD2'),('GLU','OE1'),('GLU','OE2'),('ASN','OD1'),('GLN','OE1'),('SER','OG'),('THR','OG1'),('TYR','OH'),('HIS','ND1'),('HIS','NE2'),('MET','SD')}
_POS = {('LYS','NZ'),('ARG','NH1'),('ARG','NH2'),('ARG','NE')}; _NEG={('ASP','OD1'),('ASP','OD2'),('GLU','OE1'),('GLU','OE2')}
_HPHOB = {'ALA':['CB'],'VAL':['CB','CG1','CG2'],'LEU':['CB','CG','CD1','CD2'],'ILE':['CB','CG1','CG2','CD1'],'MET':['CB','CG','CE','SD'],
          'PRO':['CB','CG','CD'],'PHE':['CB','CG','CD1','CD2','CE1','CE2','CZ'],'TYR':['CB','CG','CD1','CD2','CE1','CE2'],'TRP':['CB','CG','CD1','CD2','CE2','CE3','CZ2','CZ3','CH2'],
          'CYS':['CB'],'LYS':['CB','CG','CD'],'ARG':['CB','CG'],'THR':['CG2'],'HIS':['CB']}
_RINGS = {'PHE':[['CG','CD1','CD2','CE1','CE2','CZ']],'TYR':[['CG','CD1','CD2','CE1','CE2','CZ']],'HIS':[['CG','ND1','CD2','CE1','NE2']],
          'TRP':[['CG','CD1','CD2','NE1','CE2'],['CD2','CE2','CE3','CZ2','CZ3','CH2']]}
_METALS={'ZN','MG','MN','FE','FE2','CU','NI','CO','CA'}
_cache={}
def _rec(pdb):
    if pdb in _cache: return _cache[pdb]
    A=[]
    for l in open(WORK/f'{pdb}_receptor.pdb'):
        if l.startswith(('ATOM','HETATM')):
            el=(l[76:78].strip() or l[12:14].strip()).upper()
            if el=='H': continue
            A.append(dict(name=l[12:16].strip(),resn=l[17:20].strip(),chain=l[21],resi=int(l[22:26]),xyz=np.array([float(l[30:38]),float(l[38:46]),float(l[46:54])]),el=el,het=l.startswith('HETATM')))
    P=np.array([a['xyz'] for a in A]); _cache[pdb]=(A,P,cKDTree(P)); return _cache[pdb]
def _rl(a): return f"{a['resn']}{a['resi']}"+(f":{a['chain']}" if a['chain'].strip() else '')
def _is(a,table): return (a['resn'],a['name']) in table or ('*',a['name']) in table and not a['het']

def ligand_from_pose(molblock,smiles=None):
    m=Chem.MolFromMolBlock(molblock,removeHs=False,sanitize=False); m.UpdatePropertyCache(strict=False)
    heavy=Chem.RemoveHs(m,sanitize=False); heavy.UpdatePropertyCache(strict=False)
    if smiles:
        try: return AllChem.AssignBondOrdersFromTemplate(Chem.MolFromSmiles(smiles),heavy)
        except Exception: pass
    try: Chem.SanitizeMol(heavy)
    except Exception:
        Chem.SanitizeMol(heavy,Chem.SANITIZE_FINDRADICALS|Chem.SANITIZE_SETAROMATICITY|Chem.SANITIZE_SETCONJUGATION|Chem.SANITIZE_SETHYBRIDIZATION|Chem.SANITIZE_SYMMRINGS,catchErrors=True)
    return heavy

def detect_interactions(pdb,lig):
    A,P,tree=_rec(pdb); X=lig.GetConformer().GetPositions(); out=[]
    def add(t,i,a,d,**kw): out.append(dict(type=t,lig_idx=i,lig_atom=lig.GetAtomWithIdx(i).GetSymbol(),res=_rl(a),resn=a['resn'],resi=a['resi'],chain=a['chain'],rec_atom=a['name'],dist=round(float(d),2),**kw))
    for at in lig.GetAtoms():
        i=at.GetIdx(); s=at.GetSymbol(); x=X[i]
        nH=at.GetTotalNumHs(includeNeighbors=True); nbr=[n.GetSymbol() for n in at.GetNeighbors()]
        lig_don = s in 'NO' and nH>0
        lig_acc = (s=='O') or (s=='N' and at.GetFormalCharge()<=0 and not (nH>0 and not at.GetIsAromatic()) and not (s=='N' and at.GetDegree()+nH>=4))
        lig_pos = at.GetFormalCharge()>0 and s=='N' and not any(n.GetSymbol()=='O' and n.GetFormalCharge()<0 for n in at.GetNeighbors())
        lig_neg = at.GetFormalCharge()<0 and s=='O' and not any(n.GetFormalCharge()>0 for n in at.GetNeighbors())
        nonpolar_C = s=='C' and all(n in ('C','H','F','Cl','Br','S') for n in nbr)
        for j in tree.query_ball_point(x,6.0):
            a=A[j]; d=float(np.linalg.norm(a['xyz']-x)); resn=a['resn']
            hetero_cof = a['het'] and a['el'] in ('N','O')
            if s in 'NO' and d<=3.5 and a['el'] in ('N','O'):
                pdon=_is(a,_DON) or hetero_cof; pacc=_is(a,_ACC) or hetero_cof
                if (lig_don and pacc) or (lig_acc and pdon): add('H-bond',i,a,d,role=('lig→rec' if (lig_don and pacc) else 'rec→lig'))
            if d<=4.5 and ((lig_pos and (resn,a['name']) in _NEG) or (lig_neg and (resn,a['name']) in _POS)): add('Salt bridge',i,a,d)
            if nonpolar_C and d<=4.0 and not a['het'] and a['name'] in _HPHOB.get(resn,[]): add('Hydrophobic',i,a,d)
            if s in ('Cl','Br','I') and d<=3.6 and a['el'] in ('O','N','S'): add('Halogen/F contact',i,a,d,note='halogen bond candidate')
            if s=='F' and d<=3.3 and a['el'] in ('O','N') : add('Halogen/F contact',i,a,d,note='F···polar contact')
            if a['het'] and a['resn'] in _METALS and s in 'NOS' and d<=2.8: add('Metal/heme',i,a,d,note='metal coordination')
            if a['resn'] in ('HEM','HEC') and a['name']=='FE' and s in 'NOS' and d<=3.0: add('Metal/heme',i,a,d,note='heme-iron coordination')
    # ---- ring based: pi stacking / cation-pi ----
    ri=lig.GetRingInfo(); lrings=[r for r in ri.AtomRings() if all(lig.GetAtomWithIdx(k).GetIsAromatic() for k in r)]
    byres=defaultdict(dict)
    for a in A:
        if not a['het']: byres[(a['chain'],a['resi'],a['resn'])][a['name']]=a
    for (ch,ri_,rn),atoms in byres.items():
        for pr in _RINGS.get(rn,[]):
            if not all(n in atoms for n in pr): continue
            pc=np.mean([atoms[n]['xyz'] for n in pr],0); pn=np.linalg.svd(np.array([atoms[n]['xyz'] for n in pr])-pc)[2][2]
            for r in lrings:
                lc=X[list(r)].mean(0); ln=np.linalg.svd(X[list(r)]-lc)[2][2]; d=np.linalg.norm(pc-lc)
                if d<=5.5:
                    ang=math.degrees(math.acos(min(1,abs(float(np.dot(pn,ln))))))
                    v=(pc-lc)/d; off=math.degrees(math.acos(min(1,abs(float(np.dot(ln,v))))))   # angle of centroid vector vs ligand normal
                    kind='face-to-face' if ang<=30 and off<=40 else ('edge-to-face' if ang>=60 else None)
                    if kind: add('Pi stacking',r[0],atoms[pr[0]],d,note=kind,lig_ring=list(r))
        for nm,rn_ in (('NZ','LYS'),('CZ','ARG')):
            if rn==rn_ and nm in atoms:
                for r in lrings:
                    d=np.linalg.norm(atoms[nm]['xyz']-X[list(r)].mean(0))
                    if d<=6.0: add('Cation-pi',r[0],atoms[nm],d,lig_ring=list(r))
    df=pd.DataFrame(out)
    if len(df):   # keep the closest contact per (type, residue, ligand atom)
        df=df.sort_values('dist').drop_duplicates(['type','res','lig_idx']).reset_index(drop=True)
    return df

def pose_metrics(pdb,lig,inter,poses=None):
    A,P,tree=_rec(pdb); X=lig.GetConformer().GetPositions()
    cnt=np.array([len(tree.query_ball_point(x,4.5)) for x in X]); buried=float((cnt>=8).mean())
    polar=[a.GetIdx() for a in lig.GetAtoms() if a.GetSymbol() in 'NO']
    sat=set(inter[inter.type.isin(['H-bond','Salt bridge','Metal/heme'])].lig_idx) if len(inter) else set()
    unsat=[i for i in polar if i not in sat and cnt[i]>=8]
    m=dict(heavy_atoms=lig.GetNumHeavyAtoms(),buried_frac=round(buried,2),protein_contacts=int(len(set(j for x in X for j in tree.query_ball_point(x,4.5)))),
           n_hbond=int((inter.type=='H-bond').sum()) if len(inter) else 0,n_salt=int((inter.type=='Salt bridge').sum()) if len(inter) else 0,
           n_pi=int((inter.type.isin(['Pi stacking','Cation-pi'])).sum()) if len(inter) else 0,
           n_hydrophobic_res=int(inter[inter.type=='Hydrophobic'].res.nunique()) if len(inter) else 0,
           n_metal=int((inter.type=='Metal/heme').sum()) if len(inter) else 0,buried_unsat_polar=len(unsat))
    return m

def pose_consensus(poses,top_idx=0):
    """poses = list of (score, molblock). Fraction of the other poses that land within 2 A (symmetry-aware RMSD) of the best pose."""
    try:
        mols=[ligand_from_pose(mb) for _,mb in poses]; ref=mols[top_idx]
        r=[rdMolAlign.CalcRMS(m,ref) for k,m in enumerate(mols) if k!=top_idx]
        return dict(n_poses=len(mols),within2A=int(sum(x<=2.0 for x in r)),median_rmsd=round(float(np.median(r)),2) if r else np.nan)
    except Exception as e: return dict(n_poses=len(poses),within2A=np.nan,median_rmsd=np.nan)

def native_overlap(pdb,inter):
    nat=[np.array([float(l[30:38]),float(l[38:46]),float(l[46:54])]) for l in open(WORK/f'{pdb}_native.pdb') if l.startswith('HETATM')] if (WORK/f'{pdb}_native.pdb').exists() else []
    if not nat: return None
    A,P,tree=_rec(pdb); nres=sorted({(A[j]['resn'],A[j]['resi']) for x in nat for j in tree.query_ball_point(x,4.0) if not A[j]['het']},key=lambda t:t[1])
    cres={(r.resn,r.resi) for r in inter.itertuples()} if len(inter) else set()
    shared=[r for r in nres if r in cres]
    return dict(native_site=[f'{a}{b}' for a,b in nres],shared=[f'{a}{b}' for a,b in shared],recapitulated=round(len(shared)/max(1,len(nres)),2),
                new=[f'{a}{b}' for a,b in sorted(cres-set(nres),key=lambda t:t[1])])

_pidx={}
def find_poses(pdb,name):
    """All docking modes of one molecule on one target (index of the target's SDF outputs is built once)."""
    if pdb not in _pidx:
        d=defaultdict(list)
        for f in glob.glob(str(WORK/'dock'/f'{pdb}_*.sdf')):
            if '_native_' in f: continue
            for m in read_sdf(f):
                if m.HasProp('_Name') and m.HasProp('CNNscore'):
                    d[m.GetProp('_Name')].append((float(m.GetProp('CNNscore'))*float(m.GetProp('CNNaffinity')),Chem.MolToMolBlock(m)))
        _pidx[pdb]=d
    return sorted(_pidx[pdb].get(name,[]),key=lambda t:-t[0])

# ---------------- 2D interaction map ----------------
def plot_2d(lig,inter,title='',figsize=(10,8)):
    mol=Chem.Mol(lig); rdDepictor.Compute2DCoords(mol)
    pri=['Metal/heme','Salt bridge','H-bond','Pi stacking','Cation-pi','Halogen/F contact','Hydrophobic']
    hl={};
    if len(inter):
        for t in reversed(pri):
            for i in inter[inter.type==t].lig_idx: hl[int(i)]=tuple(int(ICOL[t][k:k+2],16)/255 for k in (1,3,5))
            if t in ('Pi stacking','Cation-pi'):
                for r in inter[inter.type==t].get('lig_ring',[]):
                    for i in (r if isinstance(r,list) else []): hl[int(i)]=tuple(int(ICOL[t][k:k+2],16)/255 for k in (1,3,5))
    W=H=520; d=rdMolDraw2D.MolDraw2DCairo(W,H); d.drawOptions().addAtomIndices=False; d.drawOptions().bondLineWidth=2
    d.DrawMolecule(mol,highlightAtoms=list(hl),highlightAtomColors=hl,highlightBonds=[]); d.FinishDrawing()
    img=plt.imread(io.BytesIO(d.GetDrawingText())); pos={i:np.array([d.GetDrawCoords(i).x,d.GetDrawCoords(i).y]) for i in range(mol.GetNumAtoms())}
    fig,ax=plt.subplots(figsize=figsize); ctr=np.array([0.0,0.0]); off=np.array([-W/2,-H/2])      # ligand image centred at (0,0)
    Lx,Ly=W/2+300,H/2+230; ax.set_xlim(-Lx,Lx); ax.set_ylim(Ly,-Ly); ax.set_aspect('equal'); ax.axis('off')
    ax.imshow(img,extent=(-W/2,W/2,H/2,-H/2),zorder=0)
    if len(inter):
        node={}
        for r in inter.itertuples():
            n=node.setdefault(r.res,dict(types=set(),atoms=[],dist=9.9)); n['types'].add(r.type); n['atoms'].append(r.lig_idx); n['dist']=min(n['dist'],r.dist)
        names=list(node); ang=[]
        for nm in names:
            c=np.mean([pos[i] for i in node[nm]['atoms']],0)+off; ang.append(math.atan2(c[1],c[0]))
        order=list(np.argsort(np.array(ang)%(2*math.pi))); a=(np.array(ang)%(2*math.pi))[order]; sep=min(0.5,2*math.pi/max(len(a),1)*0.95)
        if len(a)>1:                              # place bubbles round the ligand without overlap: cut the circle at its widest gap, sweep once, rescale if needed
            gaps=(np.roll(a,-1)-a)%(2*math.pi); cut=int(np.argmax(gaps))+1
            rot=list(range(cut,len(a)))+list(range(cut)); order=[order[i] for i in rot]; b=a[rot].copy()
            for k in range(1,len(b)):
                if b[k]<b[k-1]: b[k]+=2*math.pi
            for k in range(1,len(b)): b[k]=max(b[k],b[k-1]+sep)
            span=b[-1]-b[0]
            if span>2*math.pi-sep: b=b[0]+(b-b[0])*(2*math.pi-sep)/span
            a=b
        for k,idx in enumerate(order):
            nm=names[idx]; n=node[nm]; p=ctr+np.array([math.cos(a[k])*(W/2+190),math.sin(a[k])*(H/2+140)])
            main=[t for t in pri if t in n['types']][0]
            for t in n['types']:
                sub=inter[(inter.res==nm)&(inter.type==t)]
                if t=='Hydrophobic': sub=sub.sort_values('dist').head(1)          # one thin line per hydrophobic residue
                for ai in set(sub.lig_idx):
                    q=pos[int(ai)]+off; ax.plot([p[0],q[0]],[p[1],q[1]],color=ICOL[t],lw=2.0 if t!='Hydrophobic' else 0.9,ls='--' if t in('H-bond','Salt bridge','Metal/heme') else ('-' if t!='Hydrophobic' else ':'),alpha=.9,zorder=1)
            ax.scatter([p[0]],[p[1]],s=1500,color='white',edgecolor=ICOL[main],lw=2.5,zorder=2); ax.text(p[0],p[1],nm.replace(':',chr(10)),ha='center',va='center',fontsize=8,zorder=3)
            hb=inter[(inter.res==nm)&(inter.type=='H-bond')]
            if len(hb):
                q=pos[int(hb.iloc[0].lig_idx)]+off; ax.text((p[0]+q[0])/2,(p[1]+q[1])/2,f'{hb.iloc[0].dist:.1f} A',fontsize=7,color=ICOL['H-bond'],bbox=dict(fc='white',ec='none',pad=.5),zorder=4)
    for k,(t,c) in enumerate(ICOL.items()): ax.plot([],[],color=c,lw=3,label=t)
    ax.legend(loc='lower center',ncol=4,fontsize=8,frameon=False,bbox_to_anchor=(.5,-.03)); ax.set_title(title,fontsize=11); plt.tight_layout(); plt.show()

# ---------------- 3D view ----------------
def show_3d(pdb,molblock,inter,cut=4.5,surface=False,width=820,height=600):
    import py3Dmol
    v=py3Dmol.view(width=width,height=height)
    v.addModel((WORK/f'{pdb}_receptor.pdb').read_text(),'pdb'); v.setStyle({'model':0},{'cartoon':{'color':'#d9d9d9','opacity':.75}})
    v.addModel(molblock,'sdf'); v.setStyle({'model':1},{'stick':{'colorscheme':'greenCarbon','radius':.22}})
    v.addStyle({'model':0,'within':{'distance':cut,'sel':{'model':1}},'not':{'hetflag':True}},{'stick':{'colorscheme':'whiteCarbon','radius':.1}})
    v.addStyle({'model':0,'hetflag':True,'not':{'resn':['HOH']},'within':{'distance':cut,'sel':{'model':1}}},{'stick':{'colorscheme':'orangeCarbon','radius':.12}})   # cofactor (NADPH / heme ...)
    if surface: v.addSurface(py3Dmol.VDW,{'opacity':.25,'color':'white'},{'model':0,'within':{'distance':8,'sel':{'model':1}}})
    A,P,tree=_rec(pdb); lx=Chem.MolFromMolBlock(molblock,removeHs=False,sanitize=False).GetConformer().GetPositions(); heavy=[i for i,a in enumerate(Chem.MolFromMolBlock(molblock,removeHs=False,sanitize=False).GetAtoms()) if a.GetAtomicNum()>1]
    lig_noH=Chem.RemoveHs(Chem.MolFromMolBlock(molblock,removeHs=False,sanitize=False),sanitize=False).GetConformer().GetPositions()
    if len(inter):
        for res,g in inter.groupby('res'):
            main=[t for t in ['Metal/heme','Salt bridge','H-bond','Pi stacking','Cation-pi','Halogen/F contact','Hydrophobic'] if t in set(g.type)][0]
            sel={'model':0,'chain':g.chain.iloc[0],'resi':int(g.resi.iloc[0])}
            v.addStyle(sel,{'stick':{'color':ICOL[main],'radius':.16}}); v.addLabel(res.split(':')[0],{'fontSize':11,'backgroundColor':ICOL[main],'backgroundOpacity':.55,'fontColor':'white'},sel)
        for r in inter[inter.type.isin(['H-bond','Salt bridge','Metal/heme','Halogen/F contact'])].itertuples():
            ra=[a for a in A if a['resi']==r.resi and a['name']==r.rec_atom and a['chain']==r.chain]
            if not ra: continue
            p=ra[0]['xyz']; q=lig_noH[int(r.lig_idx)]
            v.addCylinder({'start':{'x':float(p[0]),'y':float(p[1]),'z':float(p[2])},'end':{'x':float(q[0]),'y':float(q[1]),'z':float(q[2])},'radius':.05,'color':ICOL[r.type],'dashed':True,'fromCap':1,'toCap':1})
            v.addLabel(f'{r.dist:.1f}',{'fontSize':9,'backgroundOpacity':.0,'fontColor':ICOL[r.type]},{'position':{'x':float((p[0]+q[0])/2),'y':float((p[1]+q[1])/2),'z':float((p[2]+q[2])/2)}})
    v.zoomTo({'model':1}); return v

def show_all_poses(pdb,poses,width=700,height=500):
    """Animate every docking mode (ranked) - do they converge on one binding mode?"""
    import py3Dmol
    v=py3Dmol.view(width=width,height=height); v.addModel((WORK/f'{pdb}_receptor.pdb').read_text(),'pdb'); v.setStyle({'model':0},{'cartoon':{'color':'#d9d9d9','opacity':.7}})
    cols=['#2ca02c','#1f77b4','#ff7f0e','#d62728','#9467bd','#8c564b','#e377c2','#7f7f7f','#bcbd22']
    for k,(sc,mb) in enumerate(poses[:9]):
        v.addModel(mb,'sdf'); v.setStyle({'model':k+1},{'stick':{'color':cols[k%9],'radius':.15 if k else .25}})
    v.zoomTo({'model':1}); return v

# ---------------- driver ----------------
def evaluate_pose(pdb,molblock,name='',smiles=None,poses=None,scores=None,show=True,view3d=True):
    lig=ligand_from_pose(molblock,smiles); inter=detect_interactions(pdb,lig); met=pose_metrics(pdb,lig,inter)
    if poses is None: poses=find_poses(pdb,name) if name else []
    cons=pose_consensus(poses) if len(poses)>1 else {}
    nat=native_overlap(pdb,inter); pb_ok,pb_fail=pb_check(pdb,Chem.MolFromMolBlock(molblock,removeHs=False,sanitize=False))
    summ=dict(id=name,pdb=pdb,**met,**cons,pb_valid=pb_ok,pb_failed='; '.join(pb_fail),
              native_recap=(nat or {}).get('recapitulated'),key_residues=', '.join(sorted(set(inter.res))) if len(inter) else '')
    if scores: summ.update(scores)
    if show:
        print('='*100); print(f'{name}  on  {pdb}  |  '+'  '.join(f'{k}={v}' for k,v in (scores or {}).items()))
        print(f"buried {met['buried_frac']:.0%} of ligand atoms | {met['protein_contacts']} protein atoms in contact | H-bonds {met['n_hbond']} | salt bridges {met['n_salt']} | pi {met['n_pi']} | hydrophobic residues {met['n_hydrophobic_res']} | metal/heme {met['n_metal']} | buried unsatisfied polar atoms {met['buried_unsat_polar']}")
        if cons: print(f"pose consensus: {cons['within2A']} of {cons['n_poses']-1} other modes within 2 A of the best pose (median RMSD {cons['median_rmsd']} A)  -> "+('single binding mode' if cons['within2A']>=max(1,(cons['n_poses']-1)//2) else 'scattered / uncertain binding mode'))
        print('PoseBusters:', 'PASS' if pb_ok else ('FAIL: '+'; '.join(pb_fail) if pb_ok is False else 'not run'))
        if nat: print(f"native-site residues recapitulated: {nat['recapitulated']:.0%} | shared: {' '.join(nat['shared'])} | new: {' '.join(nat['new'])} | native site not contacted: {' '.join(sorted(set(nat['native_site'])-set(nat['shared']),key=lambda s:int(re.sub(r'\D','',s) or 0)))}")
        if len(inter): display(inter.drop(columns=[c for c in ('lig_ring','chain','resn','resi') if c in inter.columns]).sort_values(['type','dist']).reset_index(drop=True))
        else: print('no interactions detected within the cut-offs')
        plot_2d(lig,inter,title=f'{name} / {pdb} - 2D interaction map (atoms coloured by interaction type)')
        if view3d:
            show_3d(pdb,molblock,inter).show()
            if len(poses)>1: print('all docking modes (green/thick = best):'); show_all_poses(pdb,poses).show()
    return summ,inter

def evaluate_top(df,n=3,smiles_map=None,view3d=True):
    """df must have id,pdb,molblock,rel,CNNscore,CNNaff,minAff. Takes each molecule's best-scoring target."""
    rows=[]; smiles_map=smiles_map or {}
    for mid in list(dict.fromkeys(df.sort_values('rel',ascending=False).id))[:n]:
        b=df[df.id==mid].sort_values('rel',ascending=False).iloc[0]
        s,_=evaluate_pose(b.pdb,b.molblock,mid,smiles_map.get(mid),scores=dict(rel=round(b.rel,2),CNNscore=round(b.CNNscore,2),CNNaff=round(b.CNNaff,2),minAff=round(b.minAff,2)),view3d=view3d)
        s['protein']=b.protein; rows.append(s)
    return pd.DataFrame(rows)

def evaluate_molecule(df,mol_id,n_targets=3,smiles=None,view3d=True):
    """Same molecule on its best n targets: where does it bind best and with which residues?"""
    rows=[]
    for b in df[df.id==mol_id].sort_values('rel',ascending=False).head(n_targets).itertuples():
        s,_=evaluate_pose(b.pdb,b.molblock,mol_id,smiles,scores=dict(rel=round(b.rel,2),CNNscore=round(b.CNNscore,2),CNNaff=round(b.CNNaff,2),minAff=round(b.minAff,2)),view3d=view3d)
        s['protein']=b.protein; rows.append(s)
    return pd.DataFrame(rows)


# ================= BATCH: every (molecule, target) pair that passes a cutoff -> tables =================
_SHORT={'H-bond':'H','Salt bridge':'S','Hydrophobic':'h','Pi stacking':'π','Cation-pi':'C','Halogen/F contact':'X','Metal/heme':'M'}
def evaluate_all(df,smiles_map=None,rel_min=None,cnn_score_min=None,cnn_aff_min=None,min_aff_max=None,top_per_target=None,max_pairs=400,run_posebusters=True,min_hbond=0):
    """Apply cutoffs to the docking table (one best pose per molecule x target), then analyse EVERY surviving pair.
    Returns: pairs (one row per molecule-target pair), contacts (long table: one row per interaction), hotspots (residue frequency per target)."""
    d=df.copy()
    if rel_min is not None: d=d[d.rel>=rel_min]
    if cnn_score_min is not None: d=d[d.CNNscore>=cnn_score_min]
    if cnn_aff_min is not None: d=d[d.CNNaff>=cnn_aff_min]
    if min_aff_max is not None: d=d[d.minAff<=min_aff_max]
    d=d.sort_values('rel',ascending=False)
    if top_per_target: d=d.groupby('pdb',group_keys=False).head(top_per_target)
    d=d.head(max_pairs); smiles_map=smiles_map or {}
    print(f'{len(d)} molecule-target pairs pass the cutoffs ({d.id.nunique()} molecules x {d.pdb.nunique()} proteins) - analysing...')
    P=[];C=[]
    for b in tqdm(d.itertuples(),total=len(d)):
        try:
            lig=ligand_from_pose(b.molblock,smiles_map.get(b.id)); inter=detect_interactions(b.pdb,lig); met=pose_metrics(b.pdb,lig,inter)
            poses=find_poses(b.pdb,b.id); cons=pose_consensus(poses) if len(poses)>1 else {}
            nat=native_overlap(b.pdb,inter)
            pb_ok,pb_fail=pb_check(b.pdb,Chem.MolFromMolBlock(b.molblock,removeHs=False,sanitize=False)) if run_posebusters else (None,[])
            hres=sorted(set(inter[inter.type.isin(['H-bond','Salt bridge','Metal/heme'])].res)) if len(inter) else []
            P.append(dict(id=b.id,pdb=b.pdb,protein=b.protein,rel=round(b.rel,2),CNNscore=round(b.CNNscore,2),CNNaff=round(b.CNNaff,2),minAff=round(b.minAff,2),**met,
                          consensus=(f"{cons['within2A']}/{cons['n_poses']-1}" if cons else ''),pb_valid=pb_ok,pb_failed='; '.join(pb_fail),
                          native_recap=(nat or {}).get('recapitulated'),polar_residues=' '.join(hres),
                          hydrophobic_residues=' '.join(sorted(set(inter[inter.type=='Hydrophobic'].res))) if len(inter) else ''))
            if len(inter): C.append(inter.assign(id=b.id,pdb=b.pdb,protein=b.protein).drop(columns=[c for c in ('lig_ring',) if c in inter.columns]))
        except Exception as e: print('  skipped',b.id,b.pdb,type(e).__name__,e)
    pairs=pd.DataFrame(P); contacts_df=pd.concat(C,ignore_index=True) if C else pd.DataFrame()
    if min_hbond and len(pairs): pairs=pairs[pairs.n_hbond>=min_hbond]
    hot=pd.DataFrame()
    if len(contacts_df):
        g=contacts_df.groupby(['pdb','protein','res','type']).id.nunique().unstack('type',fill_value=0)
        g['n_ligands']=contacts_df.groupby(['pdb','protein','res']).id.nunique()
        hot=g.reset_index().sort_values(['pdb','n_ligands'],ascending=[True,False])
    return pairs,contacts_df,hot

def residue_matrix(contacts_df,pdb,min_ligands=2,top_n=25):
    """Residues x molecules for ONE target; letters = interaction types (H H-bond, S salt bridge, h hydrophobic, π pi-stacking, C cation-pi, X halogen/F, M metal)."""
    c=contacts_df[contacts_df.pdb==pdb]
    if c.empty: return pd.DataFrame()
    keep=c.groupby('res').id.nunique(); keep=keep[keep>=min(min_ligands,c.id.nunique())].sort_values(ascending=False).head(top_n).index
    m=c[c.res.isin(keep)].groupby(['res','id']).type.agg(lambda t:''.join(sorted({_SHORT[x] for x in t}))).unstack('id',fill_value='')
    m.insert(0,'n_ligands',(m!='').sum(axis=1)); return m.sort_values('n_ligands',ascending=False)

def plot_hotspots(hot,pdb,top_n=15):
    h=hot[hot.pdb==pdb].head(top_n)
    if h.empty: return
    cols=[c for c in ICOL if c in h.columns]; ax=h.set_index('res')[cols].plot(kind='barh',stacked=True,figsize=(7,0.32*len(h)+1.5),color=[ICOL[c] for c in cols])
    ax.invert_yaxis(); ax.set_xlabel('interaction count over ligands'); ax.set_title(f"{pdb}  {h.protein.iloc[0][:45]} - residue hotspots"); plt.tight_layout(); plt.show()

In [ ]:
def stage_poses_top():
    global pose_summary
    smiles_map.update({r.id:r.smiles for r in final.itertuples()})
    pose_summary=evaluate_top(R_ALL,n=5,smiles_map=smiles_map)
    display(pose_summary.drop(columns=['pb_failed']).round(2))


def stage_poses_batch():
    global ALL, b, contacts_all, g, hotspots, pairs, pdb, ps, r
    ALL=R_ALL
    pairs,contacts_all,hotspots=evaluate_all(ALL,smiles_map,**POSE_CUTOFF)

    print('\n(1) every passing molecule x protein pair')
    display(pairs.sort_values('rel',ascending=False).reset_index(drop=True))

    print('\n(2) per-protein summary')
    if len(pairs):
        ps=pairs.groupby(['pdb','protein']).agg(n_pairs=('id','count'),n_molecules=('id','nunique'),mean_rel=('rel','mean'),best_rel=('rel','max'),mean_hbond=('n_hbond','mean'),
                                                mean_buried=('buried_frac','mean'),pb_pass=('pb_valid',lambda x:(x==True).mean()),mean_native_recap=('native_recap','mean')).round(2).reset_index()
        display(ps.sort_values('best_rel',ascending=False))

    print('\n(3) residue hotspots per protein (number of distinct ligands using each residue, by interaction type)')
    for pdb,g in (hotspots.groupby('pdb') if len(hotspots) else []):
        print('\n',pdb,g.protein.iloc[0]); display(g.drop(columns=['pdb','protein']).head(12).reset_index(drop=True)); plot_hotspots(hotspots,pdb)
        print('residue x molecule matrix (H=H-bond, S=salt bridge, h=hydrophobic, π=pi-stack, C=cation-pi, X=halogen/F, M=metal):'); display(residue_matrix(contacts_all,pdb))

    for b in (pairs.sort_values('rel',ascending=False).head(VIEW_TOP).itertuples() if VIEW_TOP else []):
        r=ALL[(ALL.id==b.id)&(ALL.pdb==b.pdb)].iloc[0]
        evaluate_pose(b.pdb,r.molblock,b.id,smiles_map.get(b.id),scores=dict(rel=b.rel,CNNscore=b.CNNscore,CNNaff=b.CNNaff,minAff=b.minAff))

In [ ]:
#@title Step 6 - FORM: pose evaluation - cut-offs, all pairs, residue hot-spots, 2D/3D viewer { display-mode: "form" }
def build_poses_form():
    with gr.Blocks(title='Step 6 - Poses') as demo:
        gr.Markdown('## Step 6 - Evaluate the docked poses\nSet the cut-offs; every molecule x protein pair that passes is analysed (H-bonds, salt bridges, pi, hydrophobic, metal), the residues involved are tabulated, and any pair can be opened in 2D and 3D.')
        with gr.Row():
            rel = gr.Slider(0, 1.5, value=POSE_CUTOFF['rel_min'], step=0.05, label='Min rel (score vs. the crystal ligand)')
            cnn = gr.Slider(0, 1, value=POSE_CUTOFF['cnn_score_min'], step=0.05, label='Min CNNscore (pose confidence)')
            hb = gr.Slider(0, 6, value=POSE_CUTOFF['min_hbond'], step=1, label='Min H-bonds')
        with gr.Row():
            caff = gr.Number(value=POSE_CUTOFF['cnn_aff_min'] or 0, label='Min CNNaffinity (pK; 6 ~ 1 µM) - 0 = off')
            maff = gr.Number(value=POSE_CUTOFF['min_aff_max'] or 0, label='Max minimizedAffinity (kcal/mol, e.g. -6) - 0 = off')
            top = gr.Slider(1, 200, value=POSE_CUTOFF['top_per_target'], step=1, label='Best N molecules per protein')
            mp = gr.Slider(10, 1000, value=POSE_CUTOFF['max_pairs'], step=10, label='Max pairs (safety cap, ~1 s per pair)')
        pbk = gr.Checkbox(POSE_CUTOFF['run_posebusters'], label='Run PoseBusters on every pair')
        run = gr.Button('Evaluate all pairs above the cut-offs', variant='primary')
        log = gr.Textbox(label='Log', lines=6, interactive=False)
        msg = gr.Markdown()
        with gr.Tab('All passing pairs'): t_pairs = gr.Dataframe(interactive=False, wrap=True)
        with gr.Tab('Per protein'): t_prot = gr.Dataframe(interactive=False, wrap=True)
        with gr.Tab('Residue hot-spots'):
            t_hot = gr.Dataframe(interactive=False, wrap=True)
            gal = gr.Gallery(label='Hot-spot plots', columns=2, height=420)
        with gr.Tab('Residue x ligand contacts'): t_con = gr.Dataframe(interactive=False, wrap=True)
        gr.Markdown('### Open one pair (2D map + 3D view + residues)')
        with gr.Row():
            sel = gr.Dropdown(choices=[], label='Molecule | protein', scale=4)
            show = gr.Button('Show 2D / 3D', variant='primary', scale=1)
        vlog = gr.Textbox(label='Pose summary', lines=7, interactive=False)
        with gr.Row():
            v2d = gr.Gallery(label='2D interaction map', columns=1, height=520)
            v3d = gr.HTML(label='3D')
        vtab = gr.Dataframe(label='Interactions (residue, atom, type, distance)', interactive=False, wrap=True)

        def on_run(rel_, cnn_, hb_, ca_, ma_, top_, mp_, pb_):
            global POSE_CUTOFF
            try: need('R_ALL')
            except Exception as e: yield str(e), f'**{e}**', *[gr.update()] * 6; return
            POSE_CUTOFF = dict(rel_min=float(rel_), cnn_score_min=float(cnn_), cnn_aff_min=(ca_ or None), min_aff_max=(ma_ or None), top_per_target=int(top_), max_pairs=int(mp_), min_hbond=int(hb_), run_posebusters=bool(pb_))
            for cap, state, _ in stream_job(stage_poses_batch):
                if state == 'running': yield cap.text(), 'Analysing poses ...', *[gr.update()] * 6; continue
                if state != 'done': yield cap.text(), f'**{state}** - see the log', *[gr.update()] * 6; continue
                if not len(pairs): yield cap.text(), '**No pair passes these cut-offs** - lower the minimum rel / CNNscore.', *[gr.update()] * 6; continue
                ps = pairs.groupby(['pdb', 'protein']).agg(n_pairs=('id', 'count'), n_molecules=('id', 'nunique'), mean_rel=('rel', 'mean'), best_rel=('rel', 'max'), mean_hbond=('n_hbond', 'mean'), mean_buried=('buried_frac', 'mean'),
                                                          pb_pass=('pb_valid', lambda x: (x == True).mean()), mean_native_recap=('native_recap', 'mean')).round(2).reset_index().sort_values('best_rel', ascending=False)
                choices = [f'{r.id} | {r.pdb}' for r in pairs.sort_values('rel', ascending=False).itertuples()]
                yield (cap.text(), f'**{len(pairs)} pairs** ({pairs.id.nunique()} molecules x {pairs.pdb.nunique()} proteins) pass the cut-offs.', show_df(pairs.sort_values('rel', ascending=False)), show_df(ps),
                       show_df(hotspots), cap.images, show_df(contacts_all), gr.update(choices=choices, value=choices[0]))
        run.click(on_run, [rel, cnn, hb, caff, maff, top, mp, pbk], [log, msg, t_pairs, t_prot, t_hot, gal, t_con, sel])

        def on_show(pick):
            if not pick: yield 'Choose a pair first.', None, '', None; return
            mid, pdb = [x.strip() for x in pick.split('|')]
            r = R_ALL[(R_ALL.id == mid) & (R_ALL.pdb == pdb)].sort_values('rel', ascending=False).iloc[0]
            sc = dict(rel=round(float(r.rel), 2), CNNscore=round(float(r.CNNscore), 2), CNNaff=round(float(r.CNNaff), 2), minAff=round(float(r.minAff), 2))
            for cap, state, res in stream_job(evaluate_pose, pdb, r.molblock, mid, smiles_map.get(mid), scores=sc):
                if state == 'running': yield cap.text(), None, '', None; continue
                html = ''.join(iframe_html(h, 520) for h in cap.html3d) or '<i>no 3D view produced</i>'
                tab = show_df(res['value'][1]) if state == 'done' and res.get('value') is not None and len(res['value'][1]) else None
                yield cap.text(), cap.images, html, tab
        show.click(on_show, sel, [vlog, v2d, v3d, vtab])
    return demo

run_or_launch('poses', build_poses_form, lambda: (stage_poses_top(), stage_poses_batch()))

## Step 7 - SAR and design of more potent analogs

In [ ]:
#@title SAR engine - any compound series: R-group decomposition, Free-Wilson, matched pairs, cliffs, enumeration { display-mode: "form" }
from rdkit.Chem import rdFMCS, rdRGroupDecomposition as rgd
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_predict
import itertools

SAR_TRANSFORMS = ['none (value is already higher = better, e.g. pIC50, docking score)',
                  'IC50 / Ki / EC50 in nM -> pIC50', 'IC50 / Ki / EC50 in uM -> pIC50',
                  'MIC / value where LOWER is better -> negative log10']

def sar_prepare(df, smiles_col, act_col, transform=SAR_TRANSFORMS[0], id_col=None):
    """-> DataFrame(id, smiles, raw, act) with act oriented so that HIGHER = BETTER."""
    d = pd.DataFrame({'smiles': df[smiles_col].astype(str), 'raw': pd.to_numeric(df[act_col], errors='coerce')})
    d['id'] = df[id_col].astype(str).values if id_col and id_col in df.columns else [f'm{i + 1}' for i in range(len(df))]
    d = d.dropna(subset=['raw']); rows = []
    for r in d.itertuples():
        m = Chem.MolFromSmiles(r.smiles)
        m = standardize(m) if m is not None else None
        if m is not None: rows.append((r.id, Chem.MolToSmiles(m), r.raw))
    d = pd.DataFrame(rows, columns=['id', 'smiles', 'raw']).drop_duplicates('smiles').reset_index(drop=True)
    t = transform
    if t.startswith('IC50') and 'nM' in t: d['act'] = 9 - np.log10(d.raw.clip(lower=1e-9))
    elif t.startswith('IC50'): d['act'] = 6 - np.log10(d.raw.clip(lower=1e-9))
    elif t.startswith('MIC'): d['act'] = -np.log10(d.raw.clip(lower=1e-9))
    else: d['act'] = d.raw
    return d


def _rlabel(s):
    """'FC(F)(F)[*:2]' -> '*C(F)(F)F'  ;  '[H][*:1]' -> 'H'"""
    if s is None: return 'H'
    m = Chem.MolFromSmiles(s)
    if m is None: return s
    if sum(1 for a in m.GetAtoms() if a.GetAtomicNum() > 1) == 0: return 'H'          # only the attachment point and hydrogens
    for a in m.GetAtoms(): a.SetAtomMapNum(0)
    return Chem.MolToSmiles(m)


def sar_decompose(d, core=None, n_mcs=60, timeout=20):
    """R-group decomposition on a common core. Returns (T, info): T has columns R1.. (display labels) + raw fragments in info['frag']."""
    mols = [Chem.MolFromSmiles(s) for s in d.smiles]
    if core and core.strip():
        c = Chem.MolFromSmarts(core.strip())
        if c is None: raise ValueError('The core is not valid SMARTS/SMILES')
        how = 'user-defined core'
    else:
        samp = mols if len(mols) <= n_mcs else [mols[i] for i in np.random.default_rng(0).choice(len(mols), n_mcs, replace=False)]
        mcs = rdFMCS.FindMCS(samp, ringMatchesRingOnly=True, completeRingsOnly=True, timeout=timeout, threshold=0.8)
        if mcs.numAtoms < 6: raise ValueError(f'No common core found (largest shared fragment: {mcs.numAtoms} atoms) - the molecules look like different series; give a core SMARTS or upload one series')
        c = Chem.MolFromSmarts(mcs.smartsString); how = f'automatic maximum common substructure ({mcs.numAtoms} atoms)'
    ps = rgd.RGroupDecompositionParameters(); ps.removeAllHydrogenRGroups = True; ps.removeHydrogensPostMatch = True
    groups, unmatched = rgd.RGroupDecompose([c], mols, asSmiles=True, options=ps)
    if not groups: raise ValueError('No molecule matched the core')
    matched = [i for i in range(len(mols)) if i not in set(unmatched)]
    F = pd.DataFrame(groups, index=matched)
    rcols = [x for x in F.columns if x.startswith('R')]
    T = pd.DataFrame({x: F[x].map(_rlabel) for x in rcols}, index=F.index)
    core_smiles = F['Core'].value_counts().index[0]
    return T, dict(core=core_smiles, how=how, n=len(d), n_matched=len(matched), unmatched=[d.id[i] for i in unmatched], frag=F[rcols], rcols=rcols)


def sar_freewilson(T, y, alpha=1.0):
    """Additive R-group model. Reference = most frequent group of each site. Returns (table, stats, model, X)."""
    ref = {c: T[c].value_counts().index[0] for c in T.columns}
    X = pd.DataFrame(index=T.index)
    for c in T.columns:
        for v, n in T[c].value_counts().items():
            if v != ref[c]: X[f'{c}|{v}'] = (T[c] == v).astype(float)
    if X.shape[1] == 0: raise ValueError('The series has no variation in its R-groups')
    mdl = Ridge(alpha=alpha).fit(X, y)
    r2 = mdl.score(X, y); q2 = np.nan
    if len(y) >= 10:
        pred = cross_val_predict(Ridge(alpha=alpha), X, y, cv=KFold(5, shuffle=True, random_state=0)); q2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    tab = pd.DataFrame([dict(site=k.split('|', 1)[0], group=k.split('|', 1)[1], contribution=mdl.coef_[i], n=int(X[k].sum())) for i, k in enumerate(X.columns)])
    tab['reference_at_site'] = tab.site.map(ref)
    return tab.sort_values('contribution', ascending=False).reset_index(drop=True), dict(r2=r2, q2=q2, intercept=float(mdl.intercept_), ref=ref, n=len(y)), mdl, X


def sar_pairs(T, y, ids, cliff=1.0):
    """Matched pairs = two molecules that differ at exactly one R site. Returns (all pairs, transformation summary, cliffs)."""
    rows = []
    for site in T.columns:
        others = [c for c in T.columns if c != site]
        groups = T.groupby(others) if others else [((), T)]
        for key, g in groups:
            if len(g) < 2: continue
            for a, b in itertools.combinations(g.index, 2):
                fa, fb = T.at[a, site], T.at[b, site]
                if fa == fb: continue
                if fa > fb: a, b, fa, fb = b, a, fb, fa
                rows.append(dict(site=site, from_group=fa, to_group=fb, id_from=ids[a], id_to=ids[b], delta=float(y[b] - y[a])))
    P = pd.DataFrame(rows)
    if P.empty: return P, P, P
    S = P.groupby(['site', 'from_group', 'to_group']).delta.agg(['mean', 'std', 'count']).reset_index().rename(columns={'mean': 'mean_delta', 'std': 'sd', 'count': 'n_pairs'})
    S = S.sort_values('mean_delta', ascending=False).reset_index(drop=True)
    C = P[P.delta.abs() >= cliff].sort_values('delta', key=lambda s: -s.abs()).reset_index(drop=True)
    return P, S, C


def sar_enumerate(T, info, fw_tab, fw_stats, top_k=3, cap=2000, known_smiles=()):
    """Combine the best R-groups of every site (plus the reference group) -> new analogs ranked by the additive model."""
    frag = info['frag']; core = Chem.MolFromSmiles(info['core'])
    if core is None: raise ValueError('core could not be rebuilt')
    store = {}                                                       # (site, label) -> original fragment smiles with attachment label
    for c in frag.columns:
        for lab, s in zip(T[c], frag[c]): store.setdefault((c, lab), s)
    contrib = {(r.site, r.group): r.contribution for r in fw_tab.itertuples()}
    opts = {}
    for site in T.columns:
        pos = [(g, v) for (s_, g), v in contrib.items() if s_ == site and v > 0]
        pos = [g for g, v in sorted(pos, key=lambda t: -t[1])[:top_k]]
        opts[site] = [fw_stats['ref'][site]] + [g for g in pos if g != fw_stats['ref'][site]]
    sites = list(T.columns); combos = list(itertools.product(*[opts[s] for s in sites]))
    pred = lambda combo: fw_stats['intercept'] + sum(contrib.get((s, g), 0.0) for s, g in zip(sites, combo))
    combos.sort(key=pred, reverse=True)
    out = []; seen = set(known_smiles)
    for combo in combos[:cap * 3]:
        m = Chem.Mol(core)
        try:
            for s, g in zip(sites, combo):
                frag_s = store.get((s, g))
                if frag_s is None: raise KeyError
                m = Chem.molzip(m, Chem.MolFromSmiles(frag_s))
            m = Chem.RemoveHs(m); Chem.SanitizeMol(m); smi = Chem.MolToSmiles(m)
        except Exception: continue
        if smi in seen: continue
        seen.add(smi); out.append(dict(smiles=smi, predicted_activity=round(pred(combo), 3), **{s: g for s, g in zip(sites, combo)}))
        if len(out) >= cap: break
    D = pd.DataFrame(out)
    if len(D): D.insert(0, 'id', [f'DES_{i + 1:04d}' for i in range(len(D))])
    return D

In [ ]:
#@title Step 7 - FORM: SAR analysis of any series + enumeration of more potent analogs { display-mode: "form" }
def read_table(path):
    p = str(path).lower()
    if p.endswith(('.sdf', '.sdf.gz')):
        import gzip
        sup = Chem.ForwardSDMolSupplier(gzip.open(path) if p.endswith('.gz') else str(path)); rows = []
        for m in sup:
            if m is None: continue
            d = {k: m.GetProp(k) for k in m.GetPropNames()}; d['smiles'] = Chem.MolToSmiles(m); d.setdefault('name', m.GetProp('_Name') if m.HasProp('_Name') else ''); rows.append(d)
        return pd.DataFrame(rows)
    if p.endswith(('.xlsx', '.xls')): return pd.read_excel(path)
    return pd.read_csv(path, sep=None, engine='python')

def build_sar_form():
    SRC = ['Upload an activity table (CSV / TSV / XLSX / SDF with a SMILES column and an activity column)', 'Docking scores of the screened molecules (mean rel over targets)']
    with gr.Blocks(title='Step 7 - SAR') as demo:
        gr.Markdown('## Step 7 - Structure-activity relationship and design of more potent analogs\nOne compound series at a time: the common core is found, the substituents (R-groups) are tabulated, an additive Free-Wilson model scores each substituent, matched pairs and activity cliffs are listed, and new analogs are built from the best substituent at each position.\n\n**Needs a series** (about 15+ molecules sharing a core). For measured IC50/MIC data upload your table; otherwise the docking scores can be used (then the model describes the *docking* SAR, not biology).')
        src = gr.Radio(SRC, value=SRC[0], label='Activity source')
        f = gr.File(label='Activity table')
        with gr.Row():
            scol = gr.Dropdown([], label='SMILES column'); acol = gr.Dropdown([], label='Activity column'); icol = gr.Dropdown([], label='Name / id column (optional)')
        with gr.Row():
            tr = gr.Dropdown(SAR_TRANSFORMS, value=SAR_TRANSFORMS[0], label='Activity unit')
            core = gr.Textbox(label='Common core as SMARTS/SMILES (blank = find it automatically)')
        with gr.Row():
            alpha = gr.Slider(0.1, 10, value=1.0, step=0.1, label='Ridge strength (higher = more conservative)')
            cliff = gr.Slider(0.3, 3, value=1.0, step=0.1, label='Activity cliff = change of at least (log units)')
            topk = gr.Slider(1, 6, value=3, step=1, label='Best substituents per position to combine')
            cap_n = gr.Slider(10, 3000, value=300, step=10, label='Max designed analogs')
        run = gr.Button('Run the SAR analysis', variant='primary')
        log = gr.Textbox(label='Log', lines=6, interactive=False)
        msg = gr.Markdown()
        with gr.Tab('Free-Wilson: contribution of every substituent'): t_fw = gr.Dataframe(interactive=False, wrap=True); p_fw = gr.Image(type='pil')
        with gr.Tab('Matched pairs'): t_mp = gr.Dataframe(interactive=False, wrap=True)
        with gr.Tab('Activity cliffs'): t_cl = gr.Dataframe(interactive=False, wrap=True)
        with gr.Tab('Designed analogs (more potent, predicted)'):
            t_en = gr.Dataframe(interactive=False, wrap=True); g_en = gr.Image(type='pil'); dl = gr.File(label='Designed analogs (CSV)', interactive=False)
            dock_b = gr.Button('Dock the top designed analogs on the selected targets (optional, slow)')
            dlog = gr.Textbox(label='Docking log', lines=4, interactive=False); dres = gr.Dataframe(label='Docked analogs', interactive=False, wrap=True)

        def on_file(fl, source):
            if source != SRC[0] or fl is None: return [gr.update()] * 3
            d = read_table(getattr(fl, 'name', fl)); cols = list(d.columns)
            guess = next((c for c in cols if c.lower() in ('smiles', 'canonical_smiles', 'smile')), cols[0])
            num = [c for c in cols if pd.api.types.is_numeric_dtype(d[c])]
            act = next((c for c in cols if any(k in c.lower() for k in ('ic50', 'ki', 'ec50', 'mic', 'activity', 'pic50', 'potency', 'value'))), num[0] if num else cols[-1])
            return gr.update(choices=cols, value=guess), gr.update(choices=cols, value=act), gr.update(choices=[''] + cols, value='')
        f.change(on_file, [f, src], [scol, acol, icol])

        def work(source, fl, sc, ac, ic, tr_, core_, al, cl, tk, cn):
            global SAR_D, SAR_T, SAR_INFO, SAR_FW, SAR_STATS, SAR_ENUM, SAR_PAIRS, SAR_SUM, SAR_CLIFFS
            if source == SRC[0]:
                if fl is None: raise ValueError('Upload an activity table first.')
                d = sar_prepare(read_table(getattr(fl, 'name', fl)), sc, ac, tr_, ic or None)
            else:
                need('R_ALL'); a = R_ALL.groupby('id').rel.mean().reset_index(); a['smiles'] = a.id.map(smiles_map)
                d = sar_prepare(a.dropna(subset=['smiles']), 'smiles', 'rel', SAR_TRANSFORMS[0], 'id')
            print(f'{len(d)} molecules with an activity value')
            if len(d) < 8: raise ValueError('Fewer than 8 molecules - too few for an SAR model.')
            T, info = sar_decompose(d, core_)
            print(f"core ({info['how']}): {info['core']}\n{info['n_matched']} of {info['n']} molecules contain the core" + (f"; not matched: {', '.join(info['unmatched'][:8])}" if info['unmatched'] else ''))
            dm = d.iloc[T.index].reset_index(drop=True); T = T.reset_index(drop=True); info['frag'] = info['frag'].reset_index(drop=True)
            y = dm.act.values
            fw, st, mdl, X = sar_freewilson(T, y, al)
            print(f"Free-Wilson: r2 = {st['r2']:.2f}, cross-validated q2 = {st['q2']:.2f}  (q2 < 0.3 means the additive model is weak: treat the designed analogs as low-confidence)")
            P, S, C = sar_pairs(T, y, dm.id.tolist(), cl)
            E = sar_enumerate(T, info, fw, st, tk, cn, known_smiles=set(dm.smiles))
            SAR_D, SAR_T, SAR_INFO, SAR_FW, SAR_STATS, SAR_ENUM, SAR_PAIRS, SAR_SUM, SAR_CLIFFS = dm, T, info, fw, st, E, P, S, C
        def on_run(*a):
            for cap, state, _ in stream_job(work, *a):
                if state == 'running': yield cap.text(), 'Working ...', *[gr.update()] * 7; continue
                if state != 'done': yield cap.text(), f'**{state}** - see the log', *[gr.update()] * 7; continue
                fw = SAR_FW; fig, ax = plt.subplots(figsize=(6, max(2.5, .28 * min(len(fw), 30)))); top = pd.concat([fw.head(12), fw.tail(8)]).drop_duplicates()
                ax.barh([f'{r.site}: {r.group}' for r in top.itertuples()][::-1], top.contribution.values[::-1], color=['#2a9d8f' if v > 0 else '#d62828' for v in top.contribution.values[::-1]]); ax.axvline(0, c='k', lw=.6)
                ax.set_xlabel('change in activity vs. the most common group of that position'); fig.tight_layout()
                E = SAR_ENUM; en = len(E)
                note = (f'**{en} designed analogs** (predicted by the additive model; not validated).' if en else '**No new analog:** the best combinations are all already in your data. Raise "best substituents per position".')
                dfile = None
                if en: dfile = str(WORK / 'designed_analogs.csv'); E.to_csv(dfile, index=False)
                yield (cap.text(), f"**Core:** `{SAR_INFO['core']}` · r2 {SAR_STATS['r2']:.2f} · q2 {SAR_STATS['q2']:.2f}. {note}", show_df(fw), fig_to_pil(fig), show_df(SAR_SUM), show_df(SAR_CLIFFS),
                       show_df(E.head(200)) if en else None, mol_grid(E.smiles, [f'{r.id} {r.predicted_activity}' for r in E.itertuples()]) if en else None, dfile)
        run.click(on_run, [src, f, scol, acol, icol, tr, core, alpha, cliff, topk, cap_n], [log, msg, t_fw, p_fw, t_mp, t_cl, t_en, g_en, dl])

        def work_dock():
            global SAR_DOCKED
            need('SAR_ENUM', 'USE_T'); lib = SAR_ENUM.head(60)[['id', 'smiles']]
            r = dock_all(lib, 'designed', USE_T[USE_T.role == 'target'])
            if not len(r): raise RuntimeError('no docking result')
            SAR_DOCKED = r.groupby('id').agg(mean_rel=('rel', 'mean'), max_rel=('rel', 'max'), n_ge1=('rel', lambda x: (x >= 1).sum())).reset_index().merge(SAR_ENUM[['id', 'smiles', 'predicted_activity']], on='id').sort_values('mean_rel', ascending=False)
        def on_dock():
            for cap, state, _ in stream_job(work_dock):
                yield cap.text(), (show_df(SAR_DOCKED) if state == 'done' else gr.update())
        dock_b.click(on_dock, None, [dlog, dres])
    return demo

run_or_launch('sar', build_sar_form)

## Step 8 - ADME

In [ ]:
#@title ADME engine - physicochemical profile, drug-likeness rules, solubility, absorption/CNS heuristics, alerts { display-mode: "form" }
from rdkit.Chem import Lipinski
import matplotlib.patches as mpatches

try: pains, brenk                                       # created in the library cell
except NameError:
    _p = FilterCatalogParams(); _p.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS); pains = FilterCatalog(_p)
    _b = FilterCatalogParams(); _b.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK); brenk = FilterCatalog(_b)
try: sascorer
except NameError: sascorer = None

ADME_NOTES = """**What this is - and is not.** Rule-based physicochemical screening from the structure alone (RDKit): the classic oral drug-likeness filters
(Lipinski, Veber, Egan, Ghose, Muegge), ESOL aqueous solubility (Delaney 2004), the Egan limits for gastro-intestinal absorption, a *heuristic* CNS/BBB rule
(TPSA <= 90, MW <= 450, HBD <= 3) and PAINS/Brenk structural alerts. These are filters that flag likely problems; they are **not** measured or modelled ADME values
(no metabolism, CYP inhibition, transporters, plasma-protein binding or toxicity). Use the optional machine-learning panel for rough numbers on those endpoints."""

def _esol(logp, mw, rotb, arom_prop):
    return 0.16 - 0.63 * logp - 0.0062 * mw + 0.066 * rotb - 0.74 * arom_prop

def _sol_class(s):
    return ('highly soluble' if s > 0 else 'very soluble' if s > -2 else 'soluble' if s > -4 else 'moderately soluble' if s > -6 else 'poorly soluble' if s > -10 else 'insoluble')

def adme_row(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    mw = Descriptors.MolWt(m); logp = Crippen.MolLogP(m); tpsa = rdMolDescriptors.CalcTPSA(m)
    hbd = rdMolDescriptors.CalcNumHBD(m); hba = rdMolDescriptors.CalcNumHBA(m); rotb = rdMolDescriptors.CalcNumRotatableBonds(m)
    rings = rdMolDescriptors.CalcNumRings(m); arings = rdMolDescriptors.CalcNumAromaticRings(m); heavy = m.GetNumHeavyAtoms()
    fsp3 = rdMolDescriptors.CalcFractionCSP3(m); mr = Crippen.MolMR(m); natoms = Chem.AddHs(m).GetNumAtoms()
    nC = sum(a.GetSymbol() == 'C' for a in m.GetAtoms()); nhet = sum(a.GetSymbol() not in ('C', 'H') for a in m.GetAtoms())
    ap = sum(a.GetIsAromatic() for a in m.GetAtoms()) / max(1, heavy)
    logS = _esol(logp, mw, rotb, ap)
    lip_v = int(mw > 500) + int(logp > 5) + int(hbd > 5) + int(hba > 10)
    r = dict(MW=mw, cLogP=logp, TPSA=tpsa, HBD=hbd, HBA=hba, RotB=rotb, Rings=rings, AromRings=arings, HeavyAtoms=heavy, Fsp3=fsp3,
             logS_ESOL=logS, solubility=_sol_class(logS), Lipinski_violations=lip_v,
             Lipinski=lip_v <= 1, Veber=(rotb <= 10 and tpsa <= 140), Egan=(tpsa <= 131.6 and logp <= 5.88),
             Ghose=(160 <= mw <= 480 and -0.4 <= logp <= 5.6 and 40 <= mr <= 130 and 20 <= natoms <= 70),
             Muegge=(200 <= mw <= 600 and -2 <= logp <= 5 and tpsa <= 150 and rings <= 7 and nC > 4 and nhet > 1 and rotb <= 15 and hba <= 10 and hbd <= 5),
             LeadLike=(250 <= mw <= 350 and logp <= 3.5 and rotb <= 7))
    r['rules_passed(5)'] = int(r['Lipinski']) + int(r['Veber']) + int(r['Egan']) + int(r['Ghose']) + int(r['Muegge'])
    r['GI_absorption'] = 'high' if r['Egan'] else 'low'
    r['CNS_BBB_heuristic'] = 'likely' if (tpsa <= 90 and mw <= 450 and hbd <= 3) else 'unlikely'
    pm = [e.GetDescription() for e in pains.GetMatches(m)]; bm = [e.GetDescription() for e in brenk.GetMatches(m)]
    r['PAINS'] = len(pm); r['Brenk'] = len(bm); r['alerts'] = '; '.join(dict.fromkeys(pm + bm))[:120]
    r['QED'] = QED.qed(m); r['SA'] = float(sascorer.calculateScore(m)) if sascorer else np.nan
    flag = 'green' if (r['rules_passed(5)'] >= 4 and not pm and logS > -6) else ('red' if (r['rules_passed(5)'] <= 2 or logS < -8) else 'amber')
    r['ADME_flag'] = flag
    return r

def adme_table(df, smiles_col='smiles', id_col='id'):
    rows = []
    for r in df.itertuples(index=False):
        d = r._asdict() if hasattr(r, '_asdict') else {}
        smi = d.get(smiles_col) if d else None
        v = adme_row(smi) if isinstance(smi, str) else None
        if v is not None: rows.append(dict(id=d.get(id_col, f'm{len(rows) + 1}'), smiles=smi, **v))
    return pd.DataFrame(rows)

def adme_summary(T):
    """pass-rate of every rule + traffic-light counts"""
    rules = ['Lipinski', 'Veber', 'Egan', 'Ghose', 'Muegge', 'LeadLike']
    s = pd.DataFrame(dict(rule=rules + ['no PAINS alert', 'solubility better than -6 (ESOL)', 'GI absorption high', 'CNS/BBB heuristic likely'],
                          pass_rate=[T[r].mean() for r in rules] + [(T.PAINS == 0).mean(), (T.logS_ESOL > -6).mean(), (T.GI_absorption == 'high').mean(), (T.CNS_BBB_heuristic == 'likely').mean()]))
    s['pass_rate'] = (100 * s.pass_rate).round(0).astype(int).astype(str) + ' %'
    return s, T.ADME_flag.value_counts().reindex(['green', 'amber', 'red']).fillna(0).astype(int)

def egg_fig(T):
    fig, ax = plt.subplots(figsize=(6.6, 5))
    ax.add_patch(mpatches.Rectangle((0, -2), 131.6, 5.88 + 2, fc='#fff3b0', ec='none', alpha=.6, label='Egan limits (GI absorption)'))
    ax.add_patch(mpatches.Rectangle((0, 0), 90, 5, fc='#ffd166', ec='none', alpha=.6, label='CNS heuristic zone (TPSA <= 90)'))
    col = T.ADME_flag.map({'green': '#2a9d8f', 'amber': '#e9a100', 'red': '#d62828'})
    ax.scatter(T.TPSA, T.cLogP, c=col, s=34, edgecolor='k', lw=.4, zorder=3)
    ax.set_xlabel('TPSA (A$^2$)'); ax.set_ylabel('cLogP (Crippen)'); ax.set_xlim(0, max(180, T.TPSA.max() + 10)); ax.set_ylim(min(-2, T.cLogP.min() - .5), max(7, T.cLogP.max() + .5))
    ax.legend(loc='lower right', fontsize=8, frameon=True); ax.set_title('Absorption / CNS map (point colour = ADME flag: green / amber / red)', fontsize=10); fig.tight_layout(); return fig

_RAD = [('LIPO\ncLogP', 'cLogP', -3, 8, -0.7, 5.0), ('SIZE\nMW', 'MW', 100, 700, 150, 500), ('POLAR\nTPSA', 'TPSA', 0, 200, 20, 130),
        ('INSOLU\nlogS', 'logS_ESOL', -10, 2, -6, 0), ('INSATU\nFsp3', 'Fsp3', 0, 1, 0.25, 1.0), ('FLEX\nRotB', 'RotB', 0, 15, 0, 9)]
def radar_fig(row):
    """bioavailability radar (axes after the SwissADME layout); green band = favourable range"""
    n = len(_RAD); ang = np.linspace(0, 2 * np.pi, n, endpoint=False); closed = lambda v: np.r_[v, v[:1]]; angc = closed(ang)
    nz = lambda v, lo, hi: float(np.clip((v - lo) / (hi - lo), 0, 1))
    inner = closed(np.array([nz(a, lo, hi) for _, _, lo, hi, a, b in _RAD])); outer = closed(np.array([nz(b, lo, hi) for _, _, lo, hi, a, b in _RAD]))
    val = closed(np.array([nz(row[k], lo, hi) for _, k, lo, hi, a, b in _RAD]))
    fig = plt.figure(figsize=(4.6, 4.6)); ax = fig.add_subplot(111, polar=True)
    ax.fill_between(angc, inner, outer, color='#9bd8c8', alpha=.6, label='favourable range'); ax.plot(angc, val, color='#c1121f', lw=2, label=str(row.get('id', 'molecule'))); ax.fill(angc, val, color='#c1121f', alpha=.15)
    ax.set_xticks(ang); ax.set_xticklabels([r[0] for r in _RAD], fontsize=8); ax.set_yticklabels([]); ax.set_ylim(0, 1)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.12), fontsize=8); fig.tight_layout(); return fig

def adme_ml(smiles):
    """OPTIONAL machine-learning ADMET endpoints through the admet-ai package (installed on demand)."""
    try:
        from admet_ai import ADMETModel
    except Exception:
        print('installing admet-ai (a few minutes)...'); sh(f"{sys.executable} -m pip -q install admet-ai", check=False)
        try:
            from admet_ai import ADMETModel
        except Exception as e:
            return None, f'admet-ai could not be installed here ({type(e).__name__}). The rule-based table above is unaffected.'
    try:
        P = ADMETModel().predict(smiles=list(smiles))
        return pd.DataFrame(P).reset_index(drop=True), 'ML predictions: indicative only (trained on public data; applicability to your series is unknown).'
    except Exception as e:
        return None, f'admet-ai failed: {type(e).__name__}: {e}'

In [ ]:
#@title Step 8 - FORM: ADME / drug-likeness analysis { display-mode: "form" }
def build_adme_form():
    SRC = ['Shortlist (step 5)', 'Library (step 3)', 'Designed analogs (step 7)', 'Upload a table / paste SMILES']
    with gr.Blocks(title='Step 8 - ADME') as demo:
        gr.Markdown('## Step 8 - ADME and drug-likeness\n' + ADME_NOTES)
        src = gr.Radio(SRC, value=SRC[0], label='Molecules to profile')
        f = gr.File(label='Table (needs a SMILES column) - only for the last option')
        paste = gr.Textbox(lines=4, label='Or paste SMILES (one per line, optional name after a space)')
        ml = gr.Checkbox(False, label='Also run the optional machine-learning ADMET panel (admet-ai; installs on first use, a few minutes)')
        run = gr.Button('Profile the molecules', variant='primary')
        log = gr.Textbox(label='Log', lines=4, interactive=False); msg = gr.Markdown()
        with gr.Row():
            t_sum = gr.Dataframe(label='Pass rate of every rule', interactive=False); egg = gr.Image(label='Absorption / CNS map', type='pil')
        t_all = gr.Dataframe(label='ADME table', interactive=False, wrap=True)
        gr.Markdown('### Bioavailability radar of one molecule')
        with gr.Row():
            sel = gr.Dropdown([], label='Molecule'); rad = gr.Image(type='pil'); st = gr.Dataframe(interactive=False)
        t_ml = gr.Dataframe(label='ML ADMET panel (indicative only)', interactive=False, wrap=True); note = gr.Markdown(); dl = gr.File(label='ADME table (CSV)', interactive=False)

        def pick_df(s, fl, p):
            if s == SRC[0]: need('final'); return final[['id', 'smiles']]
            if s == SRC[1]: need('LIB'); return LIB[['id', 'smiles']]
            if s == SRC[2]: need('SAR_ENUM'); return SAR_ENUM[['id', 'smiles']] if len(SAR_ENUM) else (_ for _ in ()).throw(ValueError('No designed analogs yet (step 7).'))
            rows = []
            if fl is not None:
                d = read_table(getattr(fl, 'name', fl)); sc = next((c for c in d.columns if c.lower() in ('smiles', 'canonical_smiles', 'smile')), d.columns[0])
                rows += [(str(r[1].get('id', r[1].get('name', f'u{i}'))), r[1][sc]) for i, r in enumerate(d.iterrows())]
            for i, l in enumerate(x for x in (p or '').splitlines() if x.strip()):
                t = l.split(); rows.append((t[1] if len(t) > 1 else f'p{i + 1}', t[0]))
            if not rows: raise ValueError('Upload a table or paste SMILES.')
            return pd.DataFrame(rows, columns=['id', 'smiles'])
        def work(s, fl, p, use_ml):
            global ADME_T, ADME_ML
            d = pick_df(s, fl, p); ADME_T = adme_table(d); print(f'{len(ADME_T)} of {len(d)} molecules profiled'); ADME_ML = None
            if use_ml: ADME_ML, ADME_MSG[0] = adme_ml(ADME_T.smiles); print(ADME_MSG[0])
        ADME_MSG = ['']
        def on_run(s, fl, p, use_ml):
            for cap, state, _ in stream_job(work, s, fl, p, use_ml):
                if state == 'running': yield cap.text(), 'Working ...', *[gr.update()] * 6; continue
                if state != 'done': yield cap.text(), f'**{state}** - see the log', *[gr.update()] * 6; continue
                S, flags = adme_summary(ADME_T); fn = str(WORK / 'adme_table.csv'); ADME_T.to_csv(fn, index=False)
                ml_t = None
                if ADME_ML is not None: ml_t = show_df(pd.concat([ADME_T[['id']].reset_index(drop=True), ADME_ML.iloc[:, :40]], axis=1))
                yield (cap.text(), f"**{len(ADME_T)} molecules:** {flags['green']} green · {flags['amber']} amber · {flags['red']} red (green = passes ≥4 of 5 oral rules, no PAINS, ESOL logS > -6).",
                       show_df(S), fig_to_pil(egg_fig(ADME_T)), show_df(ADME_T), gr.update(choices=ADME_T.id.tolist(), value=ADME_T.id.iloc[0]), ml_t, fn)
        run.click(on_run, [src, f, paste, ml], [log, msg, t_sum, egg, t_all, sel, t_ml, dl])
        def on_sel(i):
            if not i or ADME_T is None: return None, None
            r = ADME_T[ADME_T.id == i].iloc[0]
            return fig_to_pil(radar_fig(r.to_dict())), pd.DataFrame({'property': ['MW', 'cLogP', 'TPSA', 'HBD', 'HBA', 'RotB', 'logS (ESOL)', 'solubility', 'GI absorption', 'CNS (heuristic)', 'QED', 'SA', 'alerts'],
                                                                      'value': [round(r.MW, 1), round(r.cLogP, 2), round(r.TPSA, 1), r.HBD, r.HBA, r.RotB, round(r.logS_ESOL, 2), r.solubility, r.GI_absorption, r.CNS_BBB_heuristic, round(r.QED, 2), r.SA if r.SA == r.SA else '', r.alerts or '-']})
        sel.change(on_sel, sel, [rad, st])
    return demo

ADME_T = None
def adme_auto():
    global ADME_T
    ADME_T = adme_table(final[['id', 'smiles']]); print(f'ADME profile: {len(ADME_T)} shortlisted molecules'); display(ADME_T.round(2))
run_or_launch('adme', build_adme_form, adme_auto)

## Step 9 - Export

In [ ]:
def stage_export():
    global OUT, _n, xw
    for _n in ('pose_summary','pairs','contacts_all','hotspots','allagg'): globals().setdefault(_n, pd.DataFrame())
    OUT=WORK/'screening_results.xlsx'
    with pd.ExcelWriter(OUT) as xw:
        pd.DataFrame(dict(setting=['organism','taxon used','library source','library size docked','MAX_RESOLUTION','REL_CUTOFF','CNN_SCORE_CUTOFF','SAR module'],
                          value=[ORGANISM,ORG_TERM,LIBRARY_SOURCE,len(LIB),MAX_RESOLUTION,REL_CUTOFF,CNN_SCORE_CUTOFF,RUN_SAR])).to_excel(xw,sheet_name='settings',index=False)
        val.astype({c:str for c in ('center',)}).to_excel(xw,sheet_name='target_validation',index=False)
        targets.astype({c:str for c in ('ligs','lig_rscc','cofs')}).to_excel(xw,sheet_name='targets',index=False)
        LIB.to_excel(xw,sheet_name='library_docked',index=False)
        final[cols+['pb_valid','pb_failed']].to_excel(xw,sheet_name='shortlist',index=False)
        pose_summary.to_excel(xw,sheet_name='pose_analysis',index=False)
        pairs.to_excel(xw,sheet_name='pose_pairs',index=False)
        contacts_all.to_excel(xw,sheet_name='pose_contacts',index=False)
        hotspots.to_excel(xw,sheet_name='pose_hotspots',index=False)
        allagg.to_excel(xw,sheet_name='all_scored',index=False)
        if RUN_SAR and 'coef' in globals(): coef.rename('FW_contribution').to_frame().to_excel(xw,sheet_name='FreeWilson')
        R_ALL.drop(columns=['molblock']).to_excel(xw,sheet_name='per_target_scores',index=False)
        if globals().get('ADME_T') is not None: ADME_T.to_excel(xw,sheet_name='ADME',index=False)
        if globals().get('SAR_FW') is not None:
            SAR_FW.to_excel(xw,sheet_name='SAR_FreeWilson',index=False); SAR_T.assign(id=SAR_D.id.values,activity=SAR_D.act.values).to_excel(xw,sheet_name='SAR_Rgroups',index=False)
            if len(SAR_SUM): SAR_SUM.to_excel(xw,sheet_name='SAR_matched_pairs',index=False)
            if len(SAR_ENUM): SAR_ENUM.to_excel(xw,sheet_name='SAR_designed',index=False)
    print('saved to', OUT)
    return pd.ExcelFile(OUT).sheet_names

In [ ]:
#@title Step 9 - FORM: export everything to Excel { display-mode: "form" }
def build_export_form():
    with gr.Blocks(title='Step 9 - Export') as demo:
        gr.Markdown('## Step 9 - Export\nOne Excel workbook: settings, target validation, library, shortlist, pose pairs / contacts / hot-spots, and (if you ran them) SAR and ADME sheets.')
        run = gr.Button('Create the Excel workbook', variant='primary'); msg = gr.Markdown(); out = gr.File(label='screening_results.xlsx', interactive=False)
        def on_run():
            try: need('final', 'R_ALL', 'val')
            except Exception as e: return f'**{e}**', None
            r = stage_export(); return f'Workbook written ({len(r)} sheets: {", ".join(r)}).', str(WORK / 'screening_results.xlsx')
        run.click(on_run, None, [msg, out])
    return demo

run_or_launch('export', build_export_form, stage_export)

In [ ]:
#@title (Optional) all forms in one tabbed window { display-mode: "form" }
#@markdown Use this instead of the separate form cells if you prefer one window with tabs. Run the *stage cells* first; the forms in the tabs share the same data.
if not AUTO:
    _b = [build_target_form, build_validate_form, build_library_form, build_dock_form, build_shortlist_form, build_poses_form, build_sar_form, build_adme_form, build_export_form]
    launch_form('dashboard', gr.TabbedInterface([b() for b in _b], ['1 Targets', '2 Validate', '3 Library', '4 Dock', '5 Shortlist', '6 Poses', '7 SAR', '8 ADME', '9 Export'], title='Multi-target screening'), height=900)

## 14. Using the result (and what to add next)

**Reading the tables:** `rel ≥ 1` = scores at least as well as the crystal ligand of that protein; prefer molecules that do this on **several** targets (`n_ge1`), keep the native-site residues (`native_recap`), have a **PoseBusters pass**, a consistent binding mode and few buried unsatisfied polar atoms. Check `PAINS/Brenk` flags and `SA` (synthetic accessibility) before choosing compounds to test.

**Experimental funnel (limited budget):** 10-20 compounds → phenotypic assay on the organism (MIC / IC50 on parasites / virus replication / antifungal MIC) → cytotoxicity on a host cell line → selectivity index (SI ≥ 10) → enzyme assay if the predicted target is a specific protein.

**Free computational upgrades:** MM-GBSA or short OpenMM MD on the top 3-5 complexes; add a human ortholog via `OFFTARGET_PDBS` for selectivity; ADMET with SwissADME / pkCSM; AlphaFold / Boltz models for proteins without crystal structures.

**Limitations:** single receptor conformation, no explicit waters, ligand protonation not enumerated, covalent / metalloprotein chemistry only approximated, CNN affinity error ≈ 1.5 pK. Large libraries should be pre-filtered before docking.